In [1]:
Path_Base = '/Users/nsweige/Desktop/Tudo/'
Path_Downloads = '/Users/nsweige/Downloads/'
 
import os
import sys
sys.path.append(os.path.abspath(Path_Base))
from nikolibmac import *

In [2]:
data = '30-09'
sigla_mes = 'SET'
salva = True
formato_dicms = True
# path completo pro arquivo, sheet_name deve ser 'responsaveis'(padrão))
ultimo_mapeados = Path_Base + 'mapeados/05-09/05-09_TJE_Base_Acumulada_Estudantes_Beneficiários_SET_2025.xlsx'

if salva:
    # obtém caminho do script
    import pathlib
    
    path = pathlib.Path().resolve().__str__()
    path.replace('\\', '/')

    # cria uma pasta com caminho até o script/Arquivos Finais/data/
    Arquivo_Final = path + '/Arquivos_Finais/' + data + '/' 

    if not os.path.exists(Arquivo_Final):
        os.makedirs(Arquivo_Final)

# Mapeamento Auto

In [ ]:
# alunos totais, usados para procurar no cad_unico
query_alunos = f"""
SELECT UNIQUE IAT.NRO_INT_ALUNO_ESTADO
FROM PDP_SE_STG.ISE_ALUNO_TURMA IAT
INNER JOIN PDP_SE_STG.ISE_TURMA T ON T.NRO_INT_TURMA = IAT.NRO_INT_TURMA
INNER JOIN PDP_SE_STG.ISE_CALENDARIO_ESTAB CE ON CE.NRO_INT_CALEND_ESTAB = IAT.NRO_INT_CALEND_ESTAB AND CURRENT DATE BETWEEN CE.DT_INICIO_ATIV AND CE.DT_FIM_ATIV
INNER JOIN PDP_SE_STG.ISE_ESTABELECIMENTO IE ON IAT.IDT_ESTAB = IE.IDT_ESTAB AND IE.IN_ESC_PRISIONAL <> 'S' AND IE.NRO_INT_DESIGNACAO NOT IN (10, 489)
INNER JOIN PDP_SE_STG.ISE_ALUNO IA ON IAT.NRO_INT_ALUNO_ESTADO = IA.NRO_INT_ALUNO_ESTADO
WHERE IAT.NRO_INT_SITUACAO IN (1,8)
AND ( 
	(T.CD_TIPO_ENSINO IN ('I2', 'E2', 'R2') -- SE FOR EM
	AND (INTEGER((DAYS(CURRENT DATE) - DAYS(IA.DT_NASCIMENTO)) / 365.25) <= 21)) -- TEM QUE TER 21 ANOS OU MENOS
	OR 
	(T.CD_TIPO_ENSINO IN ('S2', 'E9') -- SE FOR EJA
	AND (INTEGER((DAYS(CURRENT DATE) - DAYS(IA.DT_NASCIMENTO)) / 365.25) <= 29)) -- TEM QUE TER 29 ANOS OU MENOS
	)
"""

alunos = consulta_datalake(query_alunos)
print(f'Um total de {alunos.shape[0]} alunos foram recuperados, com {alunos.shape[1]} atributos.')
display(alunos.head(5))

Um total de 265478 alunos foram recuperados, com 1 atributos.


,nro_int_aluno_estado
0,1455
1,2467
2,2518
3,2529
4,2530


In [ ]:
# alunos totais, usados para procurar no cad
query_documentos_alunos = f"""
SELECT NRO_INT_ALUNO_ESTADO, NM_ALUNO_PURO, DT_NASCIMENTO, CPF, CPF_PAI, CPF_MAE, CPF_RESPONSAVEL, NM_MAE_PURO, NR_IDENT_MAE, NM_PAI_PURO, NR_IDENT_PAI, NR_DOC_MAE, NR_DOC_PAI 
FROM PDP_SE_STG.ISE_ALUNO
WHERE NRO_INT_ALUNO_ESTADO IN ({query_alunos})
"""

documentos_alunos = consulta_datalake(query_documentos_alunos)
print(f'Um total de {documentos_alunos.shape[0]} alunos foram recuperados, com {documentos_alunos.shape[1]} atributos.')
display(documentos_alunos.head(5))

Um total de 265478 alunos foram recuperados, com 13 atributos.


,nro_int_aluno_estado,nm_aluno_puro,dt_nascimento,cpf,cpf_pai,cpf_mae,cpf_responsavel,nm_mae_puro,nr_ident_mae,nm_pai_puro,nr_ident_pai,nr_doc_mae,nr_doc_pai
0,6177876,ERIK KAIK TEIXEIRA BELARMINO,2009-09-28,05255938025,None,03338378009,03338378009,KARINE PEREIRA TEIXEIRA,6107092733,RODRIGO BELARMINO,None,None,None
1,6176125,JULIA LUCAS ASCENCO,2009-04-11,60183966023,None,91708346015,91708346015,ANA ALICE DA SILVA LUCAS,None,LEANDRO DE VARGAS ASCENCO,None,None,None
2,6176124,BRUNO DE SOUZA LEITE,2007-05-04,60092864090,None,62764519087,62764519087,CARLA REGINA TEIXEIRA DE SOUZA,None,CLAUDIO DA SILVA LEITE,None,None,None
3,6176126,KEVIN WILLIAM KUNZEL,2008-05-17,86021893034,90063791072,None,00149570007,LUCIMAR DA SILVA RANGEL,None,VAGNER KUNZEL,1067459762,None,None
4,6176127,LUCAS DA ROSA E CASTRO,2009-04-10,60130916099,None,80850839068,80850839068,ANNE CAROLINE PIA DA ROSA,1074143981,WAGNER SILVA E CASTRO,None,None,None


In [5]:
matriculas = alunos['nro_int_aluno_estado'].unique()

alunos = documentos_alunos[documentos_alunos['nro_int_aluno_estado'].isin(matriculas)].copy()
alunos

,nro_int_aluno_estado,nm_aluno_puro,dt_nascimento,cpf,cpf_pai,cpf_mae,cpf_responsavel,nm_mae_puro,nr_ident_mae,nm_pai_puro,nr_ident_pai,nr_doc_mae,nr_doc_pai
0,6177876,ERIK KAIK TEIXEIRA BELARMINO,2009-09-28,05255938025,None,03338378009,03338378009,KARINE PEREIRA TEIXEIRA,6107092733,RODRIGO BELARMINO,None,None,None
1,6176125,JULIA LUCAS ASCENCO,2009-04-11,60183966023,None,91708346015,91708346015,ANA ALICE DA SILVA LUCAS,None,LEANDRO DE VARGAS ASCENCO,None,None,None
2,6176124,BRUNO DE SOUZA LEITE,2007-05-04,60092864090,None,62764519087,62764519087,CARLA REGINA TEIXEIRA DE SOUZA,None,CLAUDIO DA SILVA LEITE,None,None,None
3,6176126,KEVIN WILLIAM KUNZEL,2008-05-17,86021893034,90063791072,None,00149570007,LUCIMAR DA SILVA RANGEL,None,VAGNER KUNZEL,1067459762,None,None
4,6176127,LUCAS DA ROSA E CASTRO,2009-04-10,60130916099,None,80850839068,80850839068,ANNE CAROLINE PIA DA ROSA,1074143981,WAGNER SILVA E CASTRO,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
265473,6981648,DIANE EMANUELI DATSCH,2010-03-28,04721220022,None,00478054041,None,ROSELEI TERESINHA DEPONTI,1092674553,MARCOS DAVI DATSCH,None,None,None
265474,6981728,VINICIUS DE OLIVEIRA DA SILVA,2009-07-24,04693835005,00280183038,01347360018,None,CECILIA DE OLIVEIRA,1099622928,RAFAEL DA SILVA,7075095112,None,None
265475,6981729,VITORIA SILVA DE OLIVEIRA,2007-07-29,06498046007,None,85586927004,None,ELAINE GONCALVES DA SILVA,None,JUARES MARCOS DE OLIVEIRA,None,None,None
265476,6981736,BRENDHA EDUARDA ANTUNES INOCENCIO,2009-12-22,04603995008,None,01168042038,None,DEISE CINARA FONTOURA ANTUNES INOCENCIO,1098344029,ALEXANDRE DE OLIVEIRA SILVEIRA INOCENCIO,None,None,None


In [6]:
alunos.isna().sum()

nro_int_aluno_estado         0
nm_aluno_puro                0
dt_nascimento                0
cpf                       1210
cpf_pai                 195108
cpf_mae                  42033
cpf_responsavel         151441
nm_mae_puro                 30
nr_ident_mae            126800
nm_pai_puro              10299
nr_ident_pai            213057
nr_doc_mae              218315
nr_doc_pai              251738
dtype: int64

In [7]:
# lê arquivo da tabela do cad unico
# fev = 2181077
df_cad = ler_cad()
df_cad = df_cad[df_cad['VLR_RENDA_MEDIA'] <= 660].drop(columns='VLR_RENDA_MEDIA')
df_cad

/Users/nsweige/Desktop/Tudo/nikolibmac.py:422: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  cad = pd.read_csv(arquivo, usecols=colunas_finais, sep=';', encoding='1252')


,COD_CPF,NOME,COD_NIS,COD_FAM,DTA_NASC,COD_PARENTESCO,NOME_MAE,NOME_PAI,RG
0,8.961796e+10,SHEILA GOMES DE OLIVEIRA,1.273489e+10,11575077,1980-09-27,1.0,RAIMUNDA PEREIRA DE OLIVEIRA,JOSE GOMES DA SILVA,00000000003411435899
1,6.014324e+10,GISELE GOMES PEREIRA,1.632547e+10,11575077,2007-06-23,3.0,SHEILA GOMES DE OLIVEIRA,ETIENE PEREIRA DA COSTA,8135061086
2,6.014324e+10,GABRIEL GOMES TORRES,2.371899e+10,11575077,2013-10-25,3.0,SHEILA GOMES DE OLIVEIRA,RAIMUNDO NONATO TORRES,00000000005135032018
7,6.019543e+10,IASMIN RAMOS CARDOSO,1.634147e+10,20816448,2008-12-22,3.0,DENISE APARECIDA MARTINEZ RAMOS,FABIO BITENCOURT CARDOSO,9137043908
8,6.768890e+10,DENISE APARECIDA MARTINEZ RAMOS,1.239475e+10,20816448,1972-04-03,1.0,VERA LUCIA RAMOS,SEBASTIAO ORLI CORREA RAMOS,2056358985
...,...,...,...,...,...,...,...,...,...
3660545,4.562098e+09,BRENDA STEFHANY SOARES XAVIER,NaN,20194210898,1999-03-21,2.0,PATRICIA SOARES,IGOR FONTANA XAVIER,6112056079
3660546,6.383206e+09,KATIA ROSA MARTINS,NaN,20194225658,2004-01-16,1.0,SANDRA GEOVANE NUNES DA ROSA,JOSE MANOEL MANTELLI MARTINS,NaN
3660547,4.068987e+09,EMILY COELHO PACHECO,2.378356e+10,20194257266,2005-10-09,1.0,LIGIANE TUCHTENHAGEN COELHO,MARCOS VINICIUS PACHECO,7123526431
3660554,1.819316e+09,SCHEILA CHARQUEIRO MUNIZ,NaN,20194288307,1988-04-10,1.0,ENI GLEDI ACUNA CHARQUEIRO MUNIZ,CLOVIS BRIAO MUNIZ,1081174813


In [8]:
# cria uma base com um cpf responsável para algum aluno que foi achado via nome ou rg e pode ter cpf nulo
base_cpf_cod_fam = df_cad.copy()

# pega o primeiro cpf não nulo em uma lista ordenada crescente por cod_parentesco
base_cpf_cod_fam = base_cpf_cod_fam.sort_values(by='COD_PARENTESCO').dropna(subset='COD_CPF')
base_cpf_cod_fam = base_cpf_cod_fam.drop_duplicates(subset='COD_FAM', keep='first')
base_cpf_cod_fam = base_cpf_cod_fam[['COD_FAM', 'COD_CPF']]
base_cpf_cod_fam['COD_CPF'] = base_cpf_cod_fam['COD_CPF'].astype("int64")
base_cpf_cod_fam

,COD_FAM,COD_CPF
0,11575077,89617959372
2202577,7176551057,75931770097
882099,4648442903,3168406023
2202575,7176544867,2777481008
882101,4648486285,62749935091
...,...,...
3653825,20179902725,93478895049
3657837,20192055640,6320234067
3659819,20187283346,60200220055
3660228,20190948434,4710718032


**Chaves PROCERGS:**

**1)**

          C.p_nom_pessoa = A.NM_ALUNO_PURO
          
          AND C.p_nom_completo_mae_pessoa = A.NM_MAE_PURO
          
          AND A.DT_NASCIMENTO = C.p_dta_nasc_pessoa
 
**2)**        

          TO_NUMBER(A.CPF_MAE) = C.p_num_cpf_pessoa
          
          AND A.NM_MAE_PURO = C.p_nom_pessoa
 
**3)**                  

          TO_NUMBER(A.CPF_PAI) = C.p_num_cpf_pessoa
          
          AND A.NM_PAI_PURO = C.p_nom_pessoa
 
**4)**             

          A.NR_IDENT_MAE = C.p_num_identidade_pessoa
          
          AND A.NM_MAE_PURO = C.p_nom_pessoa
 
**5)**                

          A.NR_IDENT_PAI = C.p_num_identidade_pessoa
          
          AND A.NM_PAI_PURO = C.p_nom_pessoa
 
**6)**           

          TO_NUMBER(A.CPF_PAI)          = C.p_num_cpf_pessoa
 
**7)**           

          TO_NUMBER(A.CPF_MAE)          = C.p_num_cpf_pessoa
          
**8)**      

          TO_NUMBER(A.CPF)  = C.p_num_cpf_pessoa  -- CPF DO ALUNO
 
**9)**            

          TO_NUMBER(A.CPF_RESPONSAVEL)  = C.p_num_cpf_pessoa

**Chave 1:** Nome, Data de nascimento e Nome da mãe

In [9]:
chave1 = alunos[['nro_int_aluno_estado', 'nm_aluno_puro', 'dt_nascimento', 'nm_mae_puro']]
chave1 = chave1.rename(columns={'nm_aluno_puro': 'NOME', 'dt_nascimento': 'DTA_NASC', 'nm_mae_puro': 'NOME_MAE'})
chave1['DTA_NASC'] = chave1['DTA_NASC'].astype(str)

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave1 = df_cad[['NOME', 'DTA_NASC', 'NOME_MAE', 'COD_CPF', 'COD_FAM']]
merge_chave1 = pd.merge(chave1, cad_chave1, how='inner')
matriculas_unicas_chave1 = len(merge_chave1['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave1}')
merge_chave1['chave'] = '1'

# separa nos cpfs que foram achados e os que não foram
merge_chave1_sem_cpf = merge_chave1[merge_chave1['COD_CPF'].isna()]
print(f'{merge_chave1_sem_cpf.shape[0]} possuiam CPF nulo')
merge_chave1_com_cpf = merge_chave1[~merge_chave1['COD_CPF'].isna()]
print(f'{merge_chave1_com_cpf.shape[0]} possuiam CPF preenchido')

# busca na base criada por cod_fam
merge_chave1_sem_cpf = merge_chave1_sem_cpf.drop(columns='COD_CPF')
merge_chave1_sem_cpf = pd.merge(merge_chave1_sem_cpf, base_cpf_cod_fam, how='left')

merge_chave1 = pd.concat([merge_chave1_sem_cpf, merge_chave1_com_cpf], axis=0).dropna(subset='COD_CPF')
display(merge_chave1)

Quantidade de matrículas únicas encontradas: 85050
310 possuiam CPF nulo
84742 possuiam CPF preenchido


,nro_int_aluno_estado,NOME,DTA_NASC,NOME_MAE,COD_FAM,chave,COD_CPF
0,6184001,DANIELY DAIANE FERNANDES DA SILVA ANHAIA,2009-07-05,DAIANE FERNANDES,4344405668,1,3.068796e+09
1,5296000,GABRIEL ANTUNES DA ROSA,2009-11-03,VIVIANE ANTUNES DA SILVA,7255363903,1,8.877741e+10
2,6191812,MAURICIO GUILHERME SCHOTT SCHERBAUM,2010-02-09,ROSELI SCHOTT,3645246851,1,5.517491e+08
3,5298520,EMILY FAGUNDES KURATHOWSKI,2010-03-22,GISELI FAGUNDES KURATHOWSKI,2534842510,1,1.754930e+08
4,6209743,EDUARDA LEITE,2009-10-05,ALINE DE OLIVEIRA DE ARAUJO,7834061186,1,1.172323e+09
...,...,...,...,...,...,...,...
85047,6981536,CRISTIAN NATANAEL DA ROSA,2010-01-12,VERONICA CRISTINA ALMEIDA DOS SANTOS,6435724067,1,6.067122e+09
85048,6981562,ANGELINA DOS SANTOS DA ROSA,2008-07-04,ANGELITA DOS SANTOS,5852827819,1,5.759651e+09
85049,6981612,MARIELLY DOMINIKE WAGNER,2010-01-11,MARILIA DOS REIS,6196640615,1,5.579556e+09
85050,6981627,EDLIN REBECA DE VARGAS,2009-07-30,RAQUEL DE VARGAS,1464856966,1,5.117193e+09


**Chave 2:** CPF da mãe e Nome da mãe

In [10]:
chave2 = alunos[['nro_int_aluno_estado', 'cpf_mae', 'nm_mae_puro']]
chave2 = chave2.rename(columns={'cpf_mae': 'COD_CPF', 'nm_mae_puro': 'NOME'})
chave2 = chave2.dropna()
chave2['COD_CPF'] = chave2['COD_CPF'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave2 = df_cad[['COD_CPF', 'NOME', 'COD_FAM']]
cad_chave2 = cad_chave2.dropna()
cad_chave2['COD_CPF'] = cad_chave2['COD_CPF'].astype("int64")
merge_chave2 = pd.merge(chave2, cad_chave2, how='inner')
matriculas_unicas_chave2 = len(merge_chave2['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave2}')
merge_chave2['chave'] = '2'

base_rf_chave2 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave2['COD_FAM'].unique())]
merge_chave2 = merge_chave2.drop(columns='COD_CPF')
merge_chave2 = pd.merge(merge_chave2, base_rf_chave2, how='left')
display(merge_chave2)

Quantidade de matrículas únicas encontradas: 69169


,nro_int_aluno_estado,NOME,COD_FAM,chave,COD_CPF
0,6176291,ELISANGELA RIBEIRO MARTINS,6169592737,2,2438125071
1,6176622,IRACEMA LUIZA WAPPLER FERSTER,2398195279,2,96609524068
2,6176620,GRAZIELA BROLL RECH,6833159866,2,952178001
3,6176621,ENIA EBERT DA ROSA,7088181029,2,72177900000
4,6176623,SIMONE LIRA DA SILVA,3049658126,2,1970451033
...,...,...,...,...,...
69164,6981536,VERONICA CRISTINA ALMEIDA DOS SANTOS,6435724067,2,1997870002
69165,6981598,MARILENE SANTOS RIBEIRO,6293166353,2,2878053010
69166,6981612,MARILIA DOS REIS,6196640615,2,2923833040
69167,6981627,RAQUEL DE VARGAS,1464856966,2,2102977005


**Chave 3:** CPF do pai e Nome do pai

In [11]:
chave3 = alunos[['nro_int_aluno_estado', 'cpf_pai', 'nm_pai_puro']]
chave3 = chave3.rename(columns={'cpf_pai': 'COD_CPF', 'nm_pai_puro': 'NOME'})
chave3 = chave3.dropna()
chave3['COD_CPF'] = chave3['COD_CPF'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave3 = df_cad[['COD_CPF', 'NOME', 'COD_FAM']]
cad_chave3 = cad_chave3.dropna()
cad_chave3['COD_CPF'] = cad_chave3['COD_CPF'].astype("int64")
merge_chave3 = pd.merge(chave3, cad_chave3, how='inner')
matriculas_unicas_chave3 = len(merge_chave3['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave3}')
merge_chave3['chave'] = '3'

base_rf_chave3 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave3['COD_FAM'].unique())]
merge_chave3 = merge_chave3.drop(columns='COD_CPF')
merge_chave3 = pd.merge(merge_chave3, base_rf_chave3, how='left')
display(merge_chave3)

Quantidade de matrículas únicas encontradas: 8483


,nro_int_aluno_estado,NOME,COD_FAM,chave,COD_CPF
0,6176291,ANSELMO GUINDANI,6169592737,3,2438125071
1,6176622,NILTON FERSTER,2398195279,3,96609524068
2,6176620,MAURICIO RECH,6833159866,3,952178001
3,6176621,CLAIR JOSE DA ROSA,7088181029,3,72177900000
4,6176623,AIRTON DA SILVA,3049658126,3,1970451033
...,...,...,...,...,...
8478,6980620,MARCELO JOBIM AGUIRRES,5673193262,3,82522634020
8479,6980973,AMANCIO RODRIGUES NETO,7696816624,3,921687010
8480,6981160,RODRIGO RODRIGUES DORNELLES SOARES,6978760048,3,83955852091
8481,6951843,NORTON NEY FERREIRA DE LEON,8759421720,3,8114306181


**Chave 4:** RG da mãe e Nome da mãe

In [12]:
# limpa nr_docs para poder usar chaves depois
def remove_nao_num(string):
    try:
        return int(float(string))
    except:
        try:
            string = re.sub('[^0-9]', '', string)
            if string == '' or len(string) > 16:
                string = np.nan
            return int(string)
        except:
            return np.nan

In [13]:
chave4 = alunos[['nro_int_aluno_estado', 'nr_ident_mae', 'nm_mae_puro']]
chave4 = chave4.rename(columns={'nr_ident_mae': 'RG', 'nm_mae_puro': 'NOME'})
chave4['RG'] = chave4['RG'].apply(remove_nao_num)
chave4 = chave4.dropna()
chave4 = chave4[chave4['RG'] < 9999999999999999]
chave4['RG'] = chave4['RG'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave4 = df_cad[['RG', 'NOME', 'COD_CPF', 'COD_FAM']]
cad_chave4['RG'] = cad_chave4['RG'].apply(remove_nao_num)
cad_chave4 = cad_chave4.dropna()
cad_chave4 = cad_chave4[cad_chave4['RG'] < 9999999999999999]
cad_chave4['RG'] = cad_chave4['RG'].astype("int64")
merge_chave4 = pd.merge(chave4, cad_chave4, how='inner')
matriculas_unicas_chave4 = len(merge_chave4['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave4}')
merge_chave4['chave'] = '4'

base_rf_chave4 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave4['COD_FAM'].unique())]
merge_chave4 = merge_chave4.drop(columns='COD_CPF')
merge_chave4 = pd.merge(merge_chave4, base_rf_chave4, how='left')
display(merge_chave4)

Quantidade de matrículas únicas encontradas: 38313


,nro_int_aluno_estado,RG,NOME,COD_FAM,chave,COD_CPF
0,6176622,1109946564,IRACEMA LUIZA WAPPLER FERSTER,2398195279,4,96609524068
1,6176620,6091619781,GRAZIELA BROLL RECH,6833159866,4,952178001
2,6176621,9041938541,ENIA EBERT DA ROSA,7088181029,4,72177900000
3,6176623,4101899328,SIMONE LIRA DA SILVA,3049658126,4,1970451033
4,6176624,8089229705,PAULA WANDER JAHN,2584778734,4,1873297025
...,...,...,...,...,...,...
38308,6981261,9093179472,LILIANE DA SILVA,4327200220,4,736085050
38309,6981273,4111705481,PATRICIA CARDOSO MINUTO,3993462602,4,2943613088
38310,6981275,7112206201,LUCIANE MARIA MAURELL GALHARDO TIBE,5463525617,4,97581119068
38311,6981301,6105528282,KAREM JEOVANA ALTMANN BARBOSA,3567169106,4,2640595067


**Chave 5:** RG do pai e Nome do pai

In [14]:
chave5 = alunos[['nro_int_aluno_estado', 'nr_ident_pai', 'nm_pai_puro']]
chave5 = chave5.rename(columns={'nr_ident_pai': 'RG', 'nm_pai_puro': 'NOME'})
chave5['RG'] = chave5['RG'].apply(remove_nao_num)
chave5 = chave5.dropna()
chave5 = chave5[chave5['RG'] < 9999999999999999]
chave5['RG'] = chave5['RG'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave5 = df_cad[['RG', 'NOME', 'COD_CPF', 'COD_FAM']]
cad_chave5['RG'] = cad_chave5['RG'].apply(remove_nao_num)
cad_chave5 = cad_chave5.dropna()
cad_chave5 = cad_chave5[cad_chave5['RG'] < 9999999999999999]
cad_chave5['RG'] = cad_chave5['RG'].astype("int64")
merge_chave5 = pd.merge(chave5, cad_chave5, how='inner')
matriculas_unicas_chave5 = len(merge_chave5['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave5}')
merge_chave5['chave'] = '5'

base_rf_chave5 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave5['COD_FAM'].unique())]
merge_chave5 = merge_chave5.drop(columns='COD_CPF')
merge_chave5 = pd.merge(merge_chave5, base_rf_chave5, how='left')
display(merge_chave5)

Quantidade de matrículas únicas encontradas: 5127


,nro_int_aluno_estado,RG,NOME,COD_FAM,chave,COD_CPF
0,6176622,1066699404,NILTON FERSTER,2398195279,5,96609524068
1,6176620,6087068091,MAURICIO RECH,6833159866,5,952178001
2,6176621,5047721625,CLAIR JOSE DA ROSA,7088181029,5,72177900000
3,6176623,1100024981,AIRTON DA SILVA,3049658126,5,1970451033
4,6176624,4088721503,MAURICIO DANIEL JAHN,2584778734,5,1873297025
...,...,...,...,...,...,...
5122,6951099,9105249883,MARCIEL GREINER,6662365505,5,2329664001
5123,6978848,5101417441,LEANDRO GASPAR DA SILVA,5214063741,5,327592079
5124,6990742,1083088276,JERONIMO ILARREGUY VARGAS,8423360016,5,82121745068
5125,6980540,5100892396,RODRIGO DOS ANJOS DE OLIVEIRA,5355924930,5,2846471010


**Chave 6:** CPF do pai

In [15]:
chave6 = alunos[['nro_int_aluno_estado', 'cpf_pai']]
chave6 = chave6.rename(columns={'cpf_pai': 'COD_CPF'})
chave6 = chave6.dropna()
chave6['COD_CPF'] = chave6['COD_CPF'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave6 = df_cad[['COD_CPF', 'COD_FAM']]
cad_chave6 = cad_chave6.dropna()
cad_chave6['COD_CPF'] = cad_chave6['COD_CPF'].astype("int64")
merge_chave6 = pd.merge(chave6, cad_chave6, how='inner')
matriculas_unicas_chave6 = len(merge_chave6['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave6}')
merge_chave6['chave'] = '6'

base_rf_chave6 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave6['COD_FAM'].unique())]
merge_chave6 = merge_chave6.drop(columns='COD_CPF')
merge_chave6 = pd.merge(merge_chave6, base_rf_chave6, how='left')
display(merge_chave6)

Quantidade de matrículas únicas encontradas: 9500


,nro_int_aluno_estado,COD_FAM,chave,COD_CPF
0,6176291,6169592737,6,2438125071
1,6176622,2398195279,6,96609524068
2,6176620,6833159866,6,952178001
3,6176621,7088181029,6,72177900000
4,6176623,3049658126,6,1970451033
...,...,...,...,...
9495,6980620,5673193262,6,82522634020
9496,6980973,7696816624,6,921687010
9497,6981160,6978760048,6,83955852091
9498,6951843,8759421720,6,8114306181


**Chave 7:** CPF da mãe

In [1]:
chave7 = alunos[['nro_int_aluno_estado', 'cpf_mae']]
chave7 = chave7.rename(columns={'cpf_mae': 'COD_CPF'})
chave7 = chave7.dropna()
chave7['COD_CPF'] = chave7['COD_CPF'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave7 = df_cad[['COD_CPF', 'COD_FAM']]
cad_chave7 = cad_chave7.dropna()
cad_chave7['COD_CPF'] = cad_chave7['COD_CPF'].astype("int64")
merge_chave7 = pd.merge(chave7, cad_chave7, how='inner')
matriculas_unicas_chave7 = len(merge_chave7['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave7}')
merge_chave7['chave'] = '7'

base_rf_chave7 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave7['COD_FAM'].unique())]
merge_chave7 = merge_chave7.drop(columns='COD_CPF')
merge_chave7 = pd.merge(merge_chave7, base_rf_chave7, how='left')
display(merge_chave7)

NameError: name 'alunos' is not defined

**Chave 8:** CPF do aluno

In [17]:
chave8 = alunos[['nro_int_aluno_estado', 'cpf']]
chave8 = chave8.rename(columns={'cpf': 'COD_CPF'})
chave8 = chave8.dropna()
chave8['COD_CPF'] = chave8['COD_CPF'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave8 = df_cad[['COD_CPF']]
cad_chave8 = cad_chave8.dropna()
cad_chave8['COD_CPF'] = cad_chave8['COD_CPF'].astype("int64")
merge_chave8 = pd.merge(chave8, cad_chave8, how='inner')
matriculas_unicas_chave8 = len(merge_chave8['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave8}')
merge_chave8['chave'] = '8'
display(merge_chave8)

Quantidade de matrículas únicas encontradas: 94602


,nro_int_aluno_estado,COD_CPF,chave
0,6176291,5896382030,8
1,6176622,4757888007,8
2,6176620,4378585022,8
3,6176621,5457643000,8
4,6176623,5801189084,8
...,...,...,...
94597,6981536,6067122081,8
94598,6981562,5759651005,8
94599,6981612,5579556020,8
94600,6981627,5117193024,8


**Chave 9:** CPF do responsável

In [18]:
chave9 = alunos[['nro_int_aluno_estado', 'cpf_responsavel']]
chave9 = chave9.rename(columns={'cpf_responsavel': 'COD_CPF'})
chave9 = chave9.dropna()
chave9['COD_CPF'] = chave9['COD_CPF'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave9 = df_cad[['COD_CPF', 'COD_FAM']]
cad_chave9 = cad_chave9.dropna()
cad_chave9['COD_CPF'] = cad_chave9['COD_CPF'].astype("int64")
merge_chave9 = pd.merge(chave9, cad_chave9, how='inner')
matriculas_unicas_chave9 = len(merge_chave9['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave9}')
merge_chave9['chave'] = '9'

base_rf_chave9 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave9['COD_FAM'].unique())]
merge_chave9 = merge_chave9.drop(columns='COD_CPF')
merge_chave9 = pd.merge(merge_chave9, base_rf_chave9, how='left')
display(merge_chave9)

Quantidade de matrículas únicas encontradas: 37493


,nro_int_aluno_estado,COD_FAM,chave,COD_CPF
0,6176622,2398195279,9,96609524068
1,6176620,6833159866,9,952178001
2,6176621,7088181029,9,72177900000
3,6176623,3049658126,9,1970451033
4,6176624,2584778734,9,1873297025
...,...,...,...,...
37488,6991519,4762467502,9,4240917042
37489,6981302,3614475239,9,1258742004
37490,6981389,3015964097,9,3412225002
37491,6981466,2703120133,9,2263363017


**Chave 10:** RG do pai e Nome do pai

In [19]:
chave10 = alunos[['nro_int_aluno_estado', 'nr_doc_pai', 'nm_pai_puro']]
chave10 = chave10.rename(columns={'nr_doc_pai': 'RG', 'nm_pai_puro': 'NOME'})
chave10['RG'] = chave10['RG'].apply(remove_nao_num)
chave10 = chave10.dropna()
chave10 = chave10[chave10['RG'] < 9999999999999999]
chave10['RG'] = chave10['RG'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave10 = df_cad[['RG', 'NOME', 'COD_CPF', 'COD_FAM']]
cad_chave10['RG'] = cad_chave10['RG'].apply(remove_nao_num)
cad_chave10 = cad_chave10.dropna()
cad_chave10 = cad_chave10[cad_chave10['RG'] < 9999999999999999]
cad_chave10['RG'] = cad_chave10['RG'].astype("int64")
merge_chave10 = pd.merge(chave10, cad_chave10, how='inner')
matriculas_unicas_chave10 = len(merge_chave10['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave10}')
merge_chave10['chave'] = '10'

base_rf_chave10 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave10['COD_FAM'].unique())]
merge_chave10 = merge_chave10.drop(columns='COD_CPF')
merge_chave10 = pd.merge(merge_chave10, base_rf_chave10, how='left')
display(merge_chave10)

Quantidade de matrículas únicas encontradas: 906


,nro_int_aluno_estado,RG,NOME,COD_FAM,chave,COD_CPF
0,6182315,9090830821,MAICON JEVERSON MARQUES LIMA,8503091070,10,1517275032
1,5295910,1061334486,MANOEL ADILIO LOPES,6897227617,10,210902035
2,5297705,9073550131,MARCOS ANTONIO SILVEIRA OLIVEIRA,7697167840,10,793287090
3,2655891,7048330125,JOSE MOACIR LEITE,4773301007,10,58404686068
4,2655901,5056108201,CLAIRTON ALMEIDA DA SILVA,4306342859,10,1228474079
...,...,...,...,...,...,...
901,6966823,8100656779,EDERSON BECHE,1950105822,10,2139050002
902,6916721,7102483919,ANDERSON DA SILVA MACHADO,7109235890,10,84360518072
903,6916963,8057016001,MARCOS LUCIANO PINTO DA SILVA,8760862963,10,93052812068
904,6919097,6087394372,CARLOS LUCIANO DE AVILA QUEIROZ,2421853206,10,3065205084


**Chave 11:** RG da mãe e Nome da mãe

In [20]:
chave11 = alunos[['nro_int_aluno_estado', 'nr_doc_mae', 'nm_mae_puro']]
chave11 = chave11.rename(columns={'nr_doc_mae': 'RG', 'nm_mae_puro': 'NOME'})
chave11['RG'] = chave11['RG'].apply(remove_nao_num)
chave11 = chave11.dropna()
chave11 = chave11[chave11['RG'] < 9999999999999999]
chave11['RG'] = chave11['RG'].astype("int64")

# faz o merge da chave, depois imprime a quantidade de matrículas única encontradas na chave
cad_chave11 = df_cad[['RG', 'NOME', 'COD_CPF', 'COD_FAM']]
cad_chave11['RG'] = cad_chave11['RG'].apply(remove_nao_num)
cad_chave11 = cad_chave11.dropna()
cad_chave11 = cad_chave11[cad_chave11['RG'] < 9999999999999999]
cad_chave11['RG'] = cad_chave11['RG'].astype("int64")
merge_chave11 = pd.merge(chave11, cad_chave11, how='inner')
matriculas_unicas_chave11 = len(merge_chave11['nro_int_aluno_estado'].unique())
print(f'Quantidade de matrículas únicas encontradas: {matriculas_unicas_chave11}')
merge_chave11['chave'] = '11'

base_rf_chave11 = base_cpf_cod_fam[base_cpf_cod_fam['COD_FAM'].isin(merge_chave11['COD_FAM'].unique())]
merge_chave11 = merge_chave11.drop(columns='COD_CPF')
merge_chave11 = pd.merge(merge_chave11, base_rf_chave11, how='left')
display(merge_chave11)

Quantidade de matrículas únicas encontradas: 11575


,nro_int_aluno_estado,RG,NOME,COD_FAM,chave,COD_CPF
0,6179314,7097794551,LUCIMARA AGRINFO,7527930128,11,1170351069
1,5293877,7053058678,SIDINEIA SOARES DE VARGAS,640268846,11,60525142053
2,5293878,6076876843,LUCIANE FAGUNDES DA SILVEIRA,2656055636,11,75183757020
3,5293947,1071534885,SEVANI SALETE DO MATTO,8822203062,11,70773467068
4,5294069,1099259151,VANESSA CONCEICAO GUEDES,8226667747,11,2786266025
...,...,...,...,...,...,...
11570,6948348,8077950007,GISLAINE SOUZA DE ABREU,5064781903,11,96146885053
11571,6948701,4093215343,JULIANA MORAES DA CONCEICAO,7729074125,11,83506730010
11572,6978476,3093112377,DIAMANTINA GARCIA DE SOUZA,6248029571,11,2531298002
11573,6951178,6095044969,LUCIANE INES KNOD,1876815701,11,868375080


**Concatenar todas as matrículas cruzadas**

In [21]:
cpfs_aluno = [merge_chave8, merge_chave1_com_cpf]
cpfs_responsavel = [merge_chave2, merge_chave3, merge_chave4, merge_chave5, merge_chave6, merge_chave7, merge_chave9, merge_chave1_sem_cpf, merge_chave10, merge_chave11]

for merge in cpfs_aluno:
    merge['ORIGEM_CPF'] = 'ALUNO'

for merge in cpfs_responsavel:
    merge['ORIGEM_CPF'] = 'RESPONSAVEL'

mapeados = pd.concat([merge_chave1_com_cpf, merge_chave8, merge_chave1_sem_cpf, merge_chave9, merge_chave2, merge_chave3, merge_chave4, merge_chave5, merge_chave6, merge_chave7, merge_chave10, merge_chave11], axis=0)
mapeados = mapeados[['nro_int_aluno_estado', 'COD_CPF', 'ORIGEM_CPF']]

origem_cpf = mapeados.drop_duplicates(subset=['nro_int_aluno_estado'], keep='first')
mapeados_primeiro_cpf = origem_cpf.drop(columns='ORIGEM_CPF')
origem_cpf

,nro_int_aluno_estado,COD_CPF,ORIGEM_CPF
0,6176291,5.896382e+09,ALUNO
1,6176622,4.757888e+09,ALUNO
2,6176620,4.378585e+09,ALUNO
3,6176621,5.457643e+09,ALUNO
4,6176623,5.801189e+09,ALUNO
...,...,...,...
10900,4930127,7.073371e+08,RESPONSAVEL
10992,4881463,3.526265e+09,RESPONSAVEL
11040,5293295,1.763855e+09,RESPONSAVEL
11109,6102081,2.046204e+09,RESPONSAVEL


In [22]:
print(str(len(origem_cpf['nro_int_aluno_estado'].unique())) + ' matrículas mapeadas') # 111958

101859 matrículas mapeadas


# Manuais e Dados Familiares

In [23]:
rotina_manual = pd.read_excel("TJE_Rotina_Manual_202510.xlsx", sheet_name='Sheet1')
rotina_manual = rotina_manual.rename(columns={'CPF_CARTAO': 'COD_CPF'}).dropna(subset='COD_CPF')
rotina_manual['MATRICULA'] = rotina_manual['MATRICULA'].astype("int64")

# remove cpfs que não aparecem no cad
rotina_manual = rotina_manual[rotina_manual['COD_CPF'].isin(df_cad['COD_CPF'].unique())]

# remove matriculas ja mapeadas
rotina_manual = rotina_manual[~rotina_manual['MATRICULA'].isin(origem_cpf['nro_int_aluno_estado'].unique())]

rotina_manual = rotina_manual[['MATRICULA', 'COD_CPF', 'ORIGEM_CPF']].sort_values(by='ORIGEM_CPF').drop_duplicates(subset='MATRICULA', keep='first')
rotina_manual

,MATRICULA,COD_CPF,ORIGEM_CPF


In [24]:
# cpfs manuais que gerariam cartão auto
rotina_manual[rotina_manual['COD_CPF'].isin(origem_cpf['COD_CPF'].unique())]

,MATRICULA,COD_CPF,ORIGEM_CPF


In [25]:
# cpfs manuais que gerariam cartão auto
origem_cpf[origem_cpf['COD_CPF'].isin(rotina_manual['COD_CPF'].unique())]

,nro_int_aluno_estado,COD_CPF,ORIGEM_CPF


In [26]:
origem_cpf_rot = rotina_manual[['COD_CPF', 'ORIGEM_CPF']]
origem_cpf_rot['ORIGEM_CPF'] = origem_cpf_rot['ORIGEM_CPF'].fillna('RESPONSAVEL')
origem_cpf_rot['ORIGEM_CPF'] = origem_cpf_rot['ORIGEM_CPF'].replace(['RESP'], ['RESPONSAVEL'])
print(origem_cpf_rot['ORIGEM_CPF'].value_counts())
origem_cpf_rot

Series([], Name: count, dtype: int64)


,COD_CPF,ORIGEM_CPF


In [27]:
origem_cpf = origem_cpf[['COD_CPF', 'ORIGEM_CPF']].drop_duplicates()
origem_cpf = pd.concat([origem_cpf, origem_cpf_rot], axis=0)
origem_cpf = origem_cpf.drop_duplicates()
origem_cpf = origem_cpf.groupby(['COD_CPF'])['ORIGEM_CPF'].apply('/'.join).reset_index()
origem_cpf = origem_cpf.drop_duplicates(subset='COD_CPF', keep='first')
origem_cpf

,COD_CPF,ORIGEM_CPF
0,1.078089e+06,RESPONSAVEL
1,2.331020e+06,RESPONSAVEL
2,5.589070e+06,RESPONSAVEL
3,5.713005e+06,RESPONSAVEL
4,5.765064e+06,RESPONSAVEL
...,...,...
101678,9.990785e+10,RESPONSAVEL
101679,9.991510e+10,RESPONSAVEL
101680,9.992356e+10,RESPONSAVEL
101681,9.995323e+10,RESPONSAVEL


In [28]:
origem_cpf['COD_CPF'] = origem_cpf['COD_CPF'].astype("int64")
origem_cpf['ORIGEM_CPF'].value_counts()

ORIGEM_CPF
ALUNO                95006
RESPONSAVEL           6665
ALUNO/RESPONSAVEL       12
Name: count, dtype: int64

In [29]:
len(origem_cpf['COD_CPF'].unique())

101683

In [30]:
base_mapeados = mapeados_primeiro_cpf[['nro_int_aluno_estado', 'COD_CPF']]
base_rm = rotina_manual[['MATRICULA', 'COD_CPF']].rename(columns={'MATRICULA': 'nro_int_aluno_estado'})

#cpfs = list(mapeados_primeiro_cpf['COD_CPF'].unique()) + list(rotina_manual['COD_CPF'].unique())
cpfs = pd.concat([base_mapeados, base_rm], axis=0).drop_duplicates(subset='nro_int_aluno_estado', keep='first')
cpfs

,nro_int_aluno_estado,COD_CPF
0,6176291,5.896382e+09
1,6176622,4.757888e+09
2,6176620,4.378585e+09
3,6176621,5.457643e+09
4,6176623,5.801189e+09
...,...,...
10900,4930127,7.073371e+08
10992,4881463,3.526265e+09
11040,5293295,1.763855e+09
11109,6102081,2.046204e+09


In [31]:
# lê arquivo da tabela do cad unico
cad_loc = ler_cad(colunas=['NOME_PESSOA', 'COD_CPF', 'COD_FAM', 'NOME_MAE', 'DTA_NASC', 'BAIRRO', 'TIPO_END', 'TITULO_END', 
                                            'NOME_END', 'NUM_END', 'COMPLEM_1', 'COMPLEM_2', 'CEP', 'CD_IBGE', 'VLR_RENDA_MEDIA', 
                                            'DDD_CONTATO1', 'NUM_TEL_CONTATO1', 'DDD_CONTATO2', 'NUM_TEL_CONTATO2', 'RG', 'COD_NIS'])
cad_loc = cad_loc.dropna(subset='COD_CPF').rename(columns={'NOME_PESSOA': 'nm_aluno_puro'})
cad_loc

/Users/nsweige/Desktop/Tudo/nikolibmac.py:422: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  cad = pd.read_csv(arquivo, usecols=colunas_finais, sep=';', encoding='1252')


,nm_aluno_puro,COD_CPF,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,TIPO_END,TITULO_END,NOME_END,NUM_END,COMPLEM_1,COMPLEM_2,CEP,CD_IBGE,VLR_RENDA_MEDIA,DDD_CONTATO1,NUM_TEL_CONTATO1,DDD_CONTATO2,NUM_TEL_CONTATO2,RG,COD_NIS
0,SHEILA GOMES DE OLIVEIRA,8.961796e+10,11575077,RAIMUNDA PEREIRA DE OLIVEIRA,1980-09-27,HARMONIA,RUA,NaN,FAGUNDES VARELA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,00000000003411435899,1.273489e+10
1,GISELE GOMES PEREIRA,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2007-06-23,HARMONIA,RUA,NaN,FAGUNDES VARELA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,8135061086,1.632547e+10
2,GABRIEL GOMES TORRES,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2013-10-25,HARMONIA,RUA,NaN,FAGUNDES VARELA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,00000000005135032018,2.371899e+10
3,ADAO SALVATO,4.712075e+10,18963854,GENOVEVA ALVES SALVATO,1946-03-16,CIDADE VERDE,RUA,NaN,NAPOLEAO TAVAREZ DE JESUS,255.0,NaN,NaN,92990000.0,4306767,1518,51.0,985472977.0,NaN,NaN,1000566107,1.059294e+10
4,MARCELO SEFNER SALVATO,8.613730e+10,18963854,MARLENE SEFNER SALVATO,1978-11-05,CIDADE VERDE,RUA,NaN,NAPOLEAO TAVAREZ DE JESUS,255.0,NaN,NaN,92990000.0,4306767,1518,51.0,985472977.0,NaN,NaN,00000000008075362502,1.604531e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3660551,FELIPE REBELLO,1.718607e+09,20194268896,BEATRIZ BUCHNER REBELLO,1991-12-08,CENTRO,AVENIDA,NaN,BOM PROGRESSO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,4099373153,NaN
3660552,VIVIANI FERRAZ DA SILVEIRA REBELLO,9.165201e+08,20194268896,VERENICE FERRAZ DA SILVEIRA,1989-06-20,CENTRO,AVENIDA,NaN,BOM PROGRESSO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,1087877971,NaN
3660553,THOR REBELLO,6.950917e+09,20194268896,VIVIANI FERRAZ DA SILVEIRA REBELLO,2022-05-08,CENTRO,AVENIDA,NaN,BOM PROGRESSO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,NaN,NaN
3660554,SCHEILA CHARQUEIRO MUNIZ,1.819316e+09,20194288307,ENI GLEDI ACUNA CHARQUEIRO MUNIZ,1988-04-10,GETULIO VARGAS,RUA,NaN,MARTIN LUTER KING,187.0,NaN,NaN,96412430.0,4301602,0,53.0,999910276.0,NaN,NaN,1081174813,NaN


In [32]:
# transforma 3 colunas do cad em coluna de logradouro e descarta 3 colunas bases
cad_loc['TIPO_END'] = cad_loc['TIPO_END'].fillna('')
cad_loc['TITULO_END'] = cad_loc['TITULO_END'].fillna('')
cad_loc['NOME_END'] = cad_loc['NOME_END'].fillna('')

cad_loc['LOGRADOURO'] = cad_loc['TIPO_END'].astype(str) + ' ' + cad_loc['TITULO_END'].astype(str) + ' ' + cad_loc['NOME_END'].astype(str)
cad_loc = cad_loc.drop(columns=['TIPO_END', 'TITULO_END', 'NOME_END'])
cad_loc

,nm_aluno_puro,COD_CPF,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,COMPLEM_1,COMPLEM_2,CEP,CD_IBGE,VLR_RENDA_MEDIA,DDD_CONTATO1,NUM_TEL_CONTATO1,DDD_CONTATO2,NUM_TEL_CONTATO2,RG,COD_NIS,LOGRADOURO
0,SHEILA GOMES DE OLIVEIRA,8.961796e+10,11575077,RAIMUNDA PEREIRA DE OLIVEIRA,1980-09-27,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,00000000003411435899,1.273489e+10,RUA FAGUNDES VARELA
1,GISELE GOMES PEREIRA,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2007-06-23,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,8135061086,1.632547e+10,RUA FAGUNDES VARELA
2,GABRIEL GOMES TORRES,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2013-10-25,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,00000000005135032018,2.371899e+10,RUA FAGUNDES VARELA
3,ADAO SALVATO,4.712075e+10,18963854,GENOVEVA ALVES SALVATO,1946-03-16,CIDADE VERDE,255.0,NaN,NaN,92990000.0,4306767,1518,51.0,985472977.0,NaN,NaN,1000566107,1.059294e+10,RUA NAPOLEAO TAVAREZ DE JESUS
4,MARCELO SEFNER SALVATO,8.613730e+10,18963854,MARLENE SEFNER SALVATO,1978-11-05,CIDADE VERDE,255.0,NaN,NaN,92990000.0,4306767,1518,51.0,985472977.0,NaN,NaN,00000000008075362502,1.604531e+10,RUA NAPOLEAO TAVAREZ DE JESUS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3660551,FELIPE REBELLO,1.718607e+09,20194268896,BEATRIZ BUCHNER REBELLO,1991-12-08,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,4099373153,NaN,AVENIDA BOM PROGRESSO
3660552,VIVIANI FERRAZ DA SILVEIRA REBELLO,9.165201e+08,20194268896,VERENICE FERRAZ DA SILVEIRA,1989-06-20,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,1087877971,NaN,AVENIDA BOM PROGRESSO
3660553,THOR REBELLO,6.950917e+09,20194268896,VIVIANI FERRAZ DA SILVEIRA REBELLO,2022-05-08,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,NaN,NaN,AVENIDA BOM PROGRESSO
3660554,SCHEILA CHARQUEIRO MUNIZ,1.819316e+09,20194288307,ENI GLEDI ACUNA CHARQUEIRO MUNIZ,1988-04-10,GETULIO VARGAS,187.0,NaN,NaN,96412430.0,4301602,0,53.0,999910276.0,NaN,NaN,1081174813,NaN,RUA MARTIN LUTER KING


In [33]:
cad_loc = cad_loc.rename(columns={'CD_IBGE': 'CIDADE'})
cad_loc['UF'] = 'RS'
cad_loc

,nm_aluno_puro,COD_CPF,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,COMPLEM_1,COMPLEM_2,CEP,CIDADE,VLR_RENDA_MEDIA,DDD_CONTATO1,NUM_TEL_CONTATO1,DDD_CONTATO2,NUM_TEL_CONTATO2,RG,COD_NIS,LOGRADOURO,UF
0,SHEILA GOMES DE OLIVEIRA,8.961796e+10,11575077,RAIMUNDA PEREIRA DE OLIVEIRA,1980-09-27,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,00000000003411435899,1.273489e+10,RUA FAGUNDES VARELA,RS
1,GISELE GOMES PEREIRA,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2007-06-23,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,8135061086,1.632547e+10,RUA FAGUNDES VARELA,RS
2,GABRIEL GOMES TORRES,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2013-10-25,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,88.0,996478458.0,NaN,NaN,00000000005135032018,2.371899e+10,RUA FAGUNDES VARELA,RS
3,ADAO SALVATO,4.712075e+10,18963854,GENOVEVA ALVES SALVATO,1946-03-16,CIDADE VERDE,255.0,NaN,NaN,92990000.0,4306767,1518,51.0,985472977.0,NaN,NaN,1000566107,1.059294e+10,RUA NAPOLEAO TAVAREZ DE JESUS,RS
4,MARCELO SEFNER SALVATO,8.613730e+10,18963854,MARLENE SEFNER SALVATO,1978-11-05,CIDADE VERDE,255.0,NaN,NaN,92990000.0,4306767,1518,51.0,985472977.0,NaN,NaN,00000000008075362502,1.604531e+10,RUA NAPOLEAO TAVAREZ DE JESUS,RS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3660551,FELIPE REBELLO,1.718607e+09,20194268896,BEATRIZ BUCHNER REBELLO,1991-12-08,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,4099373153,NaN,AVENIDA BOM PROGRESSO,RS
3660552,VIVIANI FERRAZ DA SILVEIRA REBELLO,9.165201e+08,20194268896,VERENICE FERRAZ DA SILVEIRA,1989-06-20,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,1087877971,NaN,AVENIDA BOM PROGRESSO,RS
3660553,THOR REBELLO,6.950917e+09,20194268896,VIVIANI FERRAZ DA SILVEIRA REBELLO,2022-05-08,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,55.0,999235035.0,NaN,NaN,NaN,NaN,AVENIDA BOM PROGRESSO,RS
3660554,SCHEILA CHARQUEIRO MUNIZ,1.819316e+09,20194288307,ENI GLEDI ACUNA CHARQUEIRO MUNIZ,1988-04-10,GETULIO VARGAS,187.0,NaN,NaN,96412430.0,4301602,0,53.0,999910276.0,NaN,NaN,1081174813,NaN,RUA MARTIN LUTER KING,RS


In [34]:
# concatena ddd e telefone(informações separadas no cad)
def formata_telefone(chave):
    ddd_float, num_float = chave.split(':')
    
    try:
        ddd, nada = ddd_float.split('.')
    except:
        ddd = ''

    try:
        num, nada = num_float.split('.')
    except:
        num = ''

    if (ddd != '') and (num != '') and (ddd != '0') and (num != '0'):
        return ddd + num
    elif (num != '') and (num != '0'):
        return num
    else:
        return ''

cad_loc['TELEFONE1'] = cad_loc['DDD_CONTATO1'].astype(str) + ':' + cad_loc['NUM_TEL_CONTATO1'].astype(str)
cad_loc['TELEFONE1'] = cad_loc['TELEFONE1'].apply(formata_telefone)
cad_loc['TELEFONE2'] = cad_loc['DDD_CONTATO2'].astype(str) + ':' + cad_loc['NUM_TEL_CONTATO2'].astype(str)
cad_loc['TELEFONE2'] = cad_loc['TELEFONE2'].apply(formata_telefone)
cad_loc = cad_loc.drop(columns=['DDD_CONTATO1', 'NUM_TEL_CONTATO1', 'DDD_CONTATO2', 'NUM_TEL_CONTATO2'])
cad_loc

,nm_aluno_puro,COD_CPF,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,COMPLEM_1,COMPLEM_2,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2
0,SHEILA GOMES DE OLIVEIRA,8.961796e+10,11575077,RAIMUNDA PEREIRA DE OLIVEIRA,1980-09-27,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,00000000003411435899,1.273489e+10,RUA FAGUNDES VARELA,RS,88996478458,
1,GISELE GOMES PEREIRA,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2007-06-23,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,8135061086,1.632547e+10,RUA FAGUNDES VARELA,RS,88996478458,
2,GABRIEL GOMES TORRES,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2013-10-25,HARMONIA,66.0,NaN,CASA FRENTE,92320680.0,4304606,0,00000000005135032018,2.371899e+10,RUA FAGUNDES VARELA,RS,88996478458,
3,ADAO SALVATO,4.712075e+10,18963854,GENOVEVA ALVES SALVATO,1946-03-16,CIDADE VERDE,255.0,NaN,NaN,92990000.0,4306767,1518,1000566107,1.059294e+10,RUA NAPOLEAO TAVAREZ DE JESUS,RS,51985472977,
4,MARCELO SEFNER SALVATO,8.613730e+10,18963854,MARLENE SEFNER SALVATO,1978-11-05,CIDADE VERDE,255.0,NaN,NaN,92990000.0,4306767,1518,00000000008075362502,1.604531e+10,RUA NAPOLEAO TAVAREZ DE JESUS,RS,51985472977,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3660551,FELIPE REBELLO,1.718607e+09,20194268896,BEATRIZ BUCHNER REBELLO,1991-12-08,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,4099373153,NaN,AVENIDA BOM PROGRESSO,RS,55999235035,
3660552,VIVIANI FERRAZ DA SILVEIRA REBELLO,9.165201e+08,20194268896,VERENICE FERRAZ DA SILVEIRA,1989-06-20,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,1087877971,NaN,AVENIDA BOM PROGRESSO,RS,55999235035,
3660553,THOR REBELLO,6.950917e+09,20194268896,VIVIANI FERRAZ DA SILVEIRA REBELLO,2022-05-08,CENTRO,1185.0,NaN,NaN,98575000.0,4302378,822,NaN,NaN,AVENIDA BOM PROGRESSO,RS,55999235035,
3660554,SCHEILA CHARQUEIRO MUNIZ,1.819316e+09,20194288307,ENI GLEDI ACUNA CHARQUEIRO MUNIZ,1988-04-10,GETULIO VARGAS,187.0,NaN,NaN,96412430.0,4301602,0,1081174813,NaN,RUA MARTIN LUTER KING,RS,53999910276,


In [35]:
# transforma 2 colunas do cad em uma unica coluna de complemento
cad_loc['COMPLEM_1'] = cad_loc['COMPLEM_1'].fillna('')
cad_loc['COMPLEM_2'] = cad_loc['COMPLEM_2'].fillna('')

cad_loc['COMPLEMENTO'] = cad_loc['COMPLEM_1'].astype(str) + ' ' + cad_loc['COMPLEM_2'].astype(str)
cad_loc = cad_loc.drop(columns=['COMPLEM_1', 'COMPLEM_2'])
cad_loc

,nm_aluno_puro,COD_CPF,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO
0,SHEILA GOMES DE OLIVEIRA,8.961796e+10,11575077,RAIMUNDA PEREIRA DE OLIVEIRA,1980-09-27,HARMONIA,66.0,92320680.0,4304606,0,00000000003411435899,1.273489e+10,RUA FAGUNDES VARELA,RS,88996478458,,CASA FRENTE
1,GISELE GOMES PEREIRA,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2007-06-23,HARMONIA,66.0,92320680.0,4304606,0,8135061086,1.632547e+10,RUA FAGUNDES VARELA,RS,88996478458,,CASA FRENTE
2,GABRIEL GOMES TORRES,6.014324e+10,11575077,SHEILA GOMES DE OLIVEIRA,2013-10-25,HARMONIA,66.0,92320680.0,4304606,0,00000000005135032018,2.371899e+10,RUA FAGUNDES VARELA,RS,88996478458,,CASA FRENTE
3,ADAO SALVATO,4.712075e+10,18963854,GENOVEVA ALVES SALVATO,1946-03-16,CIDADE VERDE,255.0,92990000.0,4306767,1518,1000566107,1.059294e+10,RUA NAPOLEAO TAVAREZ DE JESUS,RS,51985472977,,
4,MARCELO SEFNER SALVATO,8.613730e+10,18963854,MARLENE SEFNER SALVATO,1978-11-05,CIDADE VERDE,255.0,92990000.0,4306767,1518,00000000008075362502,1.604531e+10,RUA NAPOLEAO TAVAREZ DE JESUS,RS,51985472977,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3660551,FELIPE REBELLO,1.718607e+09,20194268896,BEATRIZ BUCHNER REBELLO,1991-12-08,CENTRO,1185.0,98575000.0,4302378,822,4099373153,NaN,AVENIDA BOM PROGRESSO,RS,55999235035,,
3660552,VIVIANI FERRAZ DA SILVEIRA REBELLO,9.165201e+08,20194268896,VERENICE FERRAZ DA SILVEIRA,1989-06-20,CENTRO,1185.0,98575000.0,4302378,822,1087877971,NaN,AVENIDA BOM PROGRESSO,RS,55999235035,,
3660553,THOR REBELLO,6.950917e+09,20194268896,VIVIANI FERRAZ DA SILVEIRA REBELLO,2022-05-08,CENTRO,1185.0,98575000.0,4302378,822,NaN,NaN,AVENIDA BOM PROGRESSO,RS,55999235035,,
3660554,SCHEILA CHARQUEIRO MUNIZ,1.819316e+09,20194288307,ENI GLEDI ACUNA CHARQUEIRO MUNIZ,1988-04-10,GETULIO VARGAS,187.0,96412430.0,4301602,0,1081174813,NaN,RUA MARTIN LUTER KING,RS,53999910276,,


In [36]:
cad_loc.isna().sum()

nm_aluno_puro            0
COD_CPF                  0
COD_FAM                  0
NOME_MAE              1740
DTA_NASC                 0
BAIRRO                   4
NUM_END             293371
CEP                      3
CIDADE                   0
VLR_RENDA_MEDIA          0
RG                 1094834
COD_NIS               4586
LOGRADOURO               0
UF                       0
TELEFONE1                0
TELEFONE2                0
COMPLEMENTO              0
dtype: int64

In [37]:
cad_aux = cad_loc[cad_loc['COD_CPF'].isin(cpfs['COD_CPF'].unique())]
responsaveis = pd.merge(cpfs, cad_aux, how='left')
responsaveis = responsaveis.dropna(subset='COD_CPF')
responsaveis

,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO
0,6176291,5.896382e+09,DANIELI VITORIA MARTINS GUINDANI,6.169593e+09,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000.0,4300802.0,300.0,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,
1,6176622,4.757888e+09,SANDRA CRISTINA FERSTER,2.398195e+09,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000.0,4307815.0,177.0,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN
2,6176620,4.378585e+09,DAVI LUCAS BROLL RECH,6.833160e+09,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000.0,4307815.0,163.0,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA
3,6176621,5.457643e+09,ILDOBRAND HUGO DA ROSA,7.088181e+09,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000.0,4307815.0,291.0,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA
4,6176623,5.801189e+09,TALITA VITORIA DA SILVA,3.049658e+09,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000.0,4307815.0,177.0,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101854,4930127,7.073371e+08,PATRICIA TEREZA DA CONCEICAO,4.904591e+09,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000.0,4305355.0,510.0,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,
101855,4881463,3.526265e+09,LUANA STEFANI DOS SANTOS,3.894383e+09,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582.0,4316808.0,100.0,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,
101856,5293295,1.763855e+09,IZIELE REZENDES DE BORTOLI,8.517294e+09,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000.0,4310603.0,0.0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS
101857,6102081,2.046204e+09,FERNANDA DE QUADRO CORREA,2.221812e+09,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720.0,4314407.0,200.0,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236


In [44]:
query_email = f"""
SELECT NRO_INT_ALUNO_ESTADO, EMAIL_EDUCAR
FROM PDP_SE_STG.ISE_ALUNO
WHERE NRO_INT_ALUNO_ESTADO IN ({query_alunos})
"""

email = consulta_datalake(query_email)
responsaveis = pd.merge(responsaveis, email, how='left')
responsaveis

,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO,email_educar
0,6176291,5.896382e+09,DANIELI VITORIA MARTINS GUINDANI,6.169593e+09,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000.0,4300802.0,300.0,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,,danieli-vmguindani@estudante.rs.gov.br
1,6176622,4.757888e+09,SANDRA CRISTINA FERSTER,2.398195e+09,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000.0,4307815.0,177.0,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN,sandra-cferster@educar.rs.gov.br
2,6176620,4.378585e+09,DAVI LUCAS BROLL RECH,6.833160e+09,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000.0,4307815.0,163.0,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA,davi-lbrech@educar.rs.gov.br
3,6176621,5.457643e+09,ILDOBRAND HUGO DA ROSA,7.088181e+09,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000.0,4307815.0,291.0,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA,ildobrand-hdrosa@educar.rs.gov.br
4,6176623,5.801189e+09,TALITA VITORIA DA SILVA,3.049658e+09,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000.0,4307815.0,177.0,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN,talita-vdsilva@educar.rs.gov.br
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,4930127,7.073371e+08,PATRICIA TEREZA DA CONCEICAO,4.904591e+09,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000.0,4305355.0,510.0,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,,emilli-dchamorra@estudante.rs.gov.br
101854,4881463,3.526265e+09,LUANA STEFANI DOS SANTOS,3.894383e+09,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582.0,4316808.0,100.0,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,,pedro-hdsdsilva5@estudante.rs.gov.br
101855,5293295,1.763855e+09,IZIELE REZENDES DE BORTOLI,8.517294e+09,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000.0,4310603.0,0.0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS,sofia-5293295@estudante.rs.gov.br
101856,6102081,2.046204e+09,FERNANDA DE QUADRO CORREA,2.221812e+09,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720.0,4314407.0,200.0,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236,adrieu-cpinheiro@estudante.rs.gov.br


In [45]:
#responsaveis = responsaveis[responsaveis['VLR_RENDA_MEDIA'] <= 660]
responsaveis

,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO,email_educar
0,6176291,5.896382e+09,DANIELI VITORIA MARTINS GUINDANI,6.169593e+09,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000.0,4300802.0,300.0,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,,danieli-vmguindani@estudante.rs.gov.br
1,6176622,4.757888e+09,SANDRA CRISTINA FERSTER,2.398195e+09,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000.0,4307815.0,177.0,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN,sandra-cferster@educar.rs.gov.br
2,6176620,4.378585e+09,DAVI LUCAS BROLL RECH,6.833160e+09,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000.0,4307815.0,163.0,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA,davi-lbrech@educar.rs.gov.br
3,6176621,5.457643e+09,ILDOBRAND HUGO DA ROSA,7.088181e+09,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000.0,4307815.0,291.0,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA,ildobrand-hdrosa@educar.rs.gov.br
4,6176623,5.801189e+09,TALITA VITORIA DA SILVA,3.049658e+09,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000.0,4307815.0,177.0,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN,talita-vdsilva@educar.rs.gov.br
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,4930127,7.073371e+08,PATRICIA TEREZA DA CONCEICAO,4.904591e+09,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000.0,4305355.0,510.0,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,,emilli-dchamorra@estudante.rs.gov.br
101854,4881463,3.526265e+09,LUANA STEFANI DOS SANTOS,3.894383e+09,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582.0,4316808.0,100.0,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,,pedro-hdsdsilva5@estudante.rs.gov.br
101855,5293295,1.763855e+09,IZIELE REZENDES DE BORTOLI,8.517294e+09,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000.0,4310603.0,0.0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS,sofia-5293295@estudante.rs.gov.br
101856,6102081,2.046204e+09,FERNANDA DE QUADRO CORREA,2.221812e+09,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720.0,4314407.0,200.0,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236,adrieu-cpinheiro@estudante.rs.gov.br


In [46]:
responsaveis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101858 entries, 0 to 101857
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   nro_int_aluno_estado  101858 non-null  int64  
 1   COD_CPF               101858 non-null  float64
 2   nm_aluno_puro         101858 non-null  object 
 3   COD_FAM               101858 non-null  float64
 4   NOME_MAE              101856 non-null  object 
 5   DTA_NASC              101858 non-null  object 
 6   BAIRRO                101858 non-null  object 
 7   NUM_END               93975 non-null   float64
 8   CEP                   101858 non-null  float64
 9   CIDADE                101858 non-null  float64
 10  VLR_RENDA_MEDIA       101858 non-null  float64
 11  RG                    57419 non-null   object 
 12  COD_NIS               101788 non-null  float64
 13  LOGRADOURO            101858 non-null  object 
 14  UF                    101858 non-null  object 
 15  

In [47]:
responsaveis.isna().sum()

nro_int_aluno_estado        0
COD_CPF                     0
nm_aluno_puro               0
COD_FAM                     0
NOME_MAE                    2
DTA_NASC                    0
BAIRRO                      0
NUM_END                  7883
CEP                         0
CIDADE                      0
VLR_RENDA_MEDIA             0
RG                      44439
COD_NIS                    70
LOGRADOURO                  0
UF                          0
TELEFONE1                   0
TELEFONE2                   0
COMPLEMENTO                 0
email_educar               17
dtype: int64

In [48]:
df_cad = ler_cad()
rf = df_cad[['COD_CPF', 'COD_FAM', 'COD_PARENTESCO']]
rf = rf[(rf['COD_PARENTESCO'] == 1) & (rf['COD_FAM'].isin(responsaveis['COD_FAM'].unique()))]
rf = rf.drop(columns='COD_PARENTESCO').rename(columns={'COD_CPF': 'CPF_RESP'})

rf = pd.merge(responsaveis, rf, how='left')
rf

/Users/nsweige/Desktop/Tudo/nikolibmac.py:422: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  cad = pd.read_csv(arquivo, usecols=colunas_finais, sep=';', encoding='1252')


,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO,email_educar,CPF_RESP
0,6176291,5.896382e+09,DANIELI VITORIA MARTINS GUINDANI,6.169593e+09,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000.0,4300802.0,300.0,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,,danieli-vmguindani@estudante.rs.gov.br,2.438125e+09
1,6176622,4.757888e+09,SANDRA CRISTINA FERSTER,2.398195e+09,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000.0,4307815.0,177.0,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN,sandra-cferster@educar.rs.gov.br,9.660952e+10
2,6176620,4.378585e+09,DAVI LUCAS BROLL RECH,6.833160e+09,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000.0,4307815.0,163.0,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA,davi-lbrech@educar.rs.gov.br,9.521780e+08
3,6176621,5.457643e+09,ILDOBRAND HUGO DA ROSA,7.088181e+09,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000.0,4307815.0,291.0,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA,ildobrand-hdrosa@educar.rs.gov.br,7.217790e+10
4,6176623,5.801189e+09,TALITA VITORIA DA SILVA,3.049658e+09,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000.0,4307815.0,177.0,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN,talita-vdsilva@educar.rs.gov.br,1.970451e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,4930127,7.073371e+08,PATRICIA TEREZA DA CONCEICAO,4.904591e+09,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000.0,4305355.0,510.0,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,,emilli-dchamorra@estudante.rs.gov.br,7.073371e+08
101854,4881463,3.526265e+09,LUANA STEFANI DOS SANTOS,3.894383e+09,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582.0,4316808.0,100.0,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,,pedro-hdsdsilva5@estudante.rs.gov.br,3.526265e+09
101855,5293295,1.763855e+09,IZIELE REZENDES DE BORTOLI,8.517294e+09,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000.0,4310603.0,0.0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS,sofia-5293295@estudante.rs.gov.br,1.763855e+09
101856,6102081,2.046204e+09,FERNANDA DE QUADRO CORREA,2.221812e+09,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720.0,4314407.0,200.0,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236,adrieu-cpinheiro@estudante.rs.gov.br,2.046204e+09


In [49]:
rf.isna().sum()

nro_int_aluno_estado        0
COD_CPF                     0
nm_aluno_puro               0
COD_FAM                     0
NOME_MAE                    2
DTA_NASC                    0
BAIRRO                      0
NUM_END                  7883
CEP                         0
CIDADE                      0
VLR_RENDA_MEDIA             0
RG                      44439
COD_NIS                    70
LOGRADOURO                  0
UF                          0
TELEFONE1                   0
TELEFONE2                   0
COMPLEMENTO                 0
email_educar               17
CPF_RESP                  522
dtype: int64

In [50]:
# lê arquivo da tabela do cad unico
cad_dados = ler_cad(colunas=['COD_CPF', 'NOME', 'DTA_NASC'])
cad_dados = cad_dados.rename(columns={'COD_CPF': 'CPF_RESP', 'NOME': 'NOME_RESP', 'DTA_NASC': 'DTA_NASC_RESP'})
cad_dados

,CPF_RESP,NOME_RESP,DTA_NASC_RESP
0,8.961796e+10,SHEILA GOMES DE OLIVEIRA,1980-09-27
1,6.014324e+10,GISELE GOMES PEREIRA,2007-06-23
2,6.014324e+10,GABRIEL GOMES TORRES,2013-10-25
3,4.712075e+10,ADAO SALVATO,1946-03-16
4,8.613730e+10,MARCELO SEFNER SALVATO,1978-11-05
...,...,...,...
3660551,1.718607e+09,FELIPE REBELLO,1991-12-08
3660552,9.165201e+08,VIVIANI FERRAZ DA SILVEIRA REBELLO,1989-06-20
3660553,6.950917e+09,THOR REBELLO,2022-05-08
3660554,1.819316e+09,SCHEILA CHARQUEIRO MUNIZ,1988-04-10


In [51]:
cad_aux = cad_dados[cad_dados['CPF_RESP'].isin(rf['CPF_RESP'].unique())].dropna(subset='CPF_RESP').drop_duplicates(subset='CPF_RESP')
cad_aux

,CPF_RESP,NOME_RESP,DTA_NASC_RESP
0,8.961796e+10,SHEILA GOMES DE OLIVEIRA,1980-09-27
8,6.768890e+10,DENISE APARECIDA MARTINEZ RAMOS,1972-04-03
24,5.808590e+10,NARA BARCELOS RODRIGUES,1970-05-14
60,7.023393e+10,MARGARETE SOUZA,1975-08-17
82,6.638640e+08,VALERIA DUMERQUE ORTIZ,1974-04-01
...,...,...,...
3660417,1.434026e+09,GLEICE MARA DE OLIVEIRA RAMOS,1986-07-25
3660442,1.912049e+09,MARIA JOCASTA FARIAS DA SILVA,1988-05-25
3660453,8.365407e+07,PAULA DIAS VIEIRA,1982-01-24
3660486,5.683644e+09,VIVIANA DANIELLY ZIMMER FARIAS,2006-10-19


In [52]:
rf = pd.merge(rf, cad_aux, how='left')
rf

,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,COD_FAM,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO,email_educar,CPF_RESP,NOME_RESP,DTA_NASC_RESP
0,6176291,5.896382e+09,DANIELI VITORIA MARTINS GUINDANI,6.169593e+09,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000.0,4300802.0,300.0,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,,danieli-vmguindani@estudante.rs.gov.br,2.438125e+09,ELISANGELA RIBEIRO MARTINS,1991-09-05
1,6176622,4.757888e+09,SANDRA CRISTINA FERSTER,2.398195e+09,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000.0,4307815.0,177.0,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN,sandra-cferster@educar.rs.gov.br,9.660952e+10,IRACEMA LUIZA WAPPLER FERSTER,1981-08-30
2,6176620,4.378585e+09,DAVI LUCAS BROLL RECH,6.833160e+09,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000.0,4307815.0,163.0,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA,davi-lbrech@educar.rs.gov.br,9.521780e+08,GRAZIELA BROLL RECH,1986-07-14
3,6176621,5.457643e+09,ILDOBRAND HUGO DA ROSA,7.088181e+09,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000.0,4307815.0,291.0,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA,ildobrand-hdrosa@educar.rs.gov.br,7.217790e+10,ENIA EBERT DA ROSA,1970-09-29
4,6176623,5.801189e+09,TALITA VITORIA DA SILVA,3.049658e+09,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000.0,4307815.0,177.0,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN,talita-vdsilva@educar.rs.gov.br,1.970451e+09,SIMONE LIRA DA SILVA,1989-04-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,4930127,7.073371e+08,PATRICIA TEREZA DA CONCEICAO,4.904591e+09,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000.0,4305355.0,510.0,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,,emilli-dchamorra@estudante.rs.gov.br,7.073371e+08,PATRICIA TEREZA DA CONCEICAO,1984-06-26
101854,4881463,3.526265e+09,LUANA STEFANI DOS SANTOS,3.894383e+09,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582.0,4316808.0,100.0,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,,pedro-hdsdsilva5@estudante.rs.gov.br,3.526265e+09,LUANA STEFANI DOS SANTOS,1994-02-25
101855,5293295,1.763855e+09,IZIELE REZENDES DE BORTOLI,8.517294e+09,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000.0,4310603.0,0.0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS,sofia-5293295@estudante.rs.gov.br,1.763855e+09,IZIELE REZENDES DE BORTOLI,1988-04-26
101856,6102081,2.046204e+09,FERNANDA DE QUADRO CORREA,2.221812e+09,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720.0,4314407.0,200.0,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236,adrieu-cpinheiro@estudante.rs.gov.br,2.046204e+09,FERNANDA DE QUADRO CORREA,1985-07-03


In [53]:
rf.isna().sum()

nro_int_aluno_estado        0
COD_CPF                     0
nm_aluno_puro               0
COD_FAM                     0
NOME_MAE                    2
DTA_NASC                    0
BAIRRO                      0
NUM_END                  7883
CEP                         0
CIDADE                      0
VLR_RENDA_MEDIA             0
RG                      44439
COD_NIS                    70
LOGRADOURO                  0
UF                          0
TELEFONE1                   0
TELEFONE2                   0
COMPLEMENTO                 0
email_educar               17
CPF_RESP                  522
NOME_RESP                 522
DTA_NASC_RESP             522
dtype: int64

In [54]:
rf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101858 entries, 0 to 101857
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   nro_int_aluno_estado  101858 non-null  int64  
 1   COD_CPF               101858 non-null  float64
 2   nm_aluno_puro         101858 non-null  object 
 3   COD_FAM               101858 non-null  float64
 4   NOME_MAE              101856 non-null  object 
 5   DTA_NASC              101858 non-null  object 
 6   BAIRRO                101858 non-null  object 
 7   NUM_END               93975 non-null   float64
 8   CEP                   101858 non-null  float64
 9   CIDADE                101858 non-null  float64
 10  VLR_RENDA_MEDIA       101858 non-null  float64
 11  RG                    57419 non-null   object 
 12  COD_NIS               101788 non-null  float64
 13  LOGRADOURO            101858 non-null  object 
 14  UF                    101858 non-null  object 
 15  

In [55]:
def float_to_txt(float):
    try:
        int, aaaaaaa = float.split('.')
        return int
    except:
        return np.nan

rf = rf.drop(columns='COD_FAM')
rf['CEP'] = rf['CEP'].fillna(0).astype("int64")
rf['COD_CPF'] = rf['COD_CPF'].astype("int64")
rf['VLR_RENDA_MEDIA'] = rf['VLR_RENDA_MEDIA'].astype("int64")
rf

,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO,email_educar,CPF_RESP,NOME_RESP,DTA_NASC_RESP
0,6176291,5896382030,DANIELI VITORIA MARTINS GUINDANI,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000,4300802.0,300,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,,danieli-vmguindani@estudante.rs.gov.br,2.438125e+09,ELISANGELA RIBEIRO MARTINS,1991-09-05
1,6176622,4757888007,SANDRA CRISTINA FERSTER,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000,4307815.0,177,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN,sandra-cferster@educar.rs.gov.br,9.660952e+10,IRACEMA LUIZA WAPPLER FERSTER,1981-08-30
2,6176620,4378585022,DAVI LUCAS BROLL RECH,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000,4307815.0,163,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA,davi-lbrech@educar.rs.gov.br,9.521780e+08,GRAZIELA BROLL RECH,1986-07-14
3,6176621,5457643000,ILDOBRAND HUGO DA ROSA,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000,4307815.0,291,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA,ildobrand-hdrosa@educar.rs.gov.br,7.217790e+10,ENIA EBERT DA ROSA,1970-09-29
4,6176623,5801189084,TALITA VITORIA DA SILVA,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000,4307815.0,177,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN,talita-vdsilva@educar.rs.gov.br,1.970451e+09,SIMONE LIRA DA SILVA,1989-04-17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,4930127,707337089,PATRICIA TEREZA DA CONCEICAO,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000,4305355.0,510,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,,emilli-dchamorra@estudante.rs.gov.br,7.073371e+08,PATRICIA TEREZA DA CONCEICAO,1984-06-26
101854,4881463,3526265003,LUANA STEFANI DOS SANTOS,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582,4316808.0,100,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,,pedro-hdsdsilva5@estudante.rs.gov.br,3.526265e+09,LUANA STEFANI DOS SANTOS,1994-02-25
101855,5293295,1763855090,IZIELE REZENDES DE BORTOLI,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000,4310603.0,0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS,sofia-5293295@estudante.rs.gov.br,1.763855e+09,IZIELE REZENDES DE BORTOLI,1988-04-26
101856,6102081,2046204069,FERNANDA DE QUADRO CORREA,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720,4314407.0,200,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236,adrieu-cpinheiro@estudante.rs.gov.br,2.046204e+09,FERNANDA DE QUADRO CORREA,1985-07-03


In [56]:
def int_to_cpf(int):
    cpf = str(int)
    if(cpf != 'nan'):
        return cpf.zfill(11)
    else:
        return ''

rf['CPF_RESP'] = rf['CPF_RESP'].astype(str).apply(float_to_txt)
rf['CPF_RESP'] = rf['CPF_RESP'].apply(int_to_cpf)

rf = pd.merge(rf, origem_cpf, how='left')
rf['COD_CPF'] = rf['COD_CPF'].apply(int_to_cpf)
rf

,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO,email_educar,CPF_RESP,NOME_RESP,DTA_NASC_RESP,ORIGEM_CPF
0,6176291,05896382030,DANIELI VITORIA MARTINS GUINDANI,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000,4300802.0,300,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,,danieli-vmguindani@estudante.rs.gov.br,02438125071,ELISANGELA RIBEIRO MARTINS,1991-09-05,ALUNO
1,6176622,04757888007,SANDRA CRISTINA FERSTER,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000,4307815.0,177,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN,sandra-cferster@educar.rs.gov.br,96609524068,IRACEMA LUIZA WAPPLER FERSTER,1981-08-30,ALUNO
2,6176620,04378585022,DAVI LUCAS BROLL RECH,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000,4307815.0,163,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA,davi-lbrech@educar.rs.gov.br,00952178001,GRAZIELA BROLL RECH,1986-07-14,ALUNO
3,6176621,05457643000,ILDOBRAND HUGO DA ROSA,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000,4307815.0,291,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA,ildobrand-hdrosa@educar.rs.gov.br,72177900000,ENIA EBERT DA ROSA,1970-09-29,ALUNO
4,6176623,05801189084,TALITA VITORIA DA SILVA,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000,4307815.0,177,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN,talita-vdsilva@educar.rs.gov.br,01970451033,SIMONE LIRA DA SILVA,1989-04-17,ALUNO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,4930127,00707337089,PATRICIA TEREZA DA CONCEICAO,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000,4305355.0,510,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,,emilli-dchamorra@estudante.rs.gov.br,00707337089,PATRICIA TEREZA DA CONCEICAO,1984-06-26,RESPONSAVEL
101854,4881463,03526265003,LUANA STEFANI DOS SANTOS,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582,4316808.0,100,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,,pedro-hdsdsilva5@estudante.rs.gov.br,03526265003,LUANA STEFANI DOS SANTOS,1994-02-25,RESPONSAVEL
101855,5293295,01763855090,IZIELE REZENDES DE BORTOLI,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000,4310603.0,0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS,sofia-5293295@estudante.rs.gov.br,01763855090,IZIELE REZENDES DE BORTOLI,1988-04-26,RESPONSAVEL
101856,6102081,02046204069,FERNANDA DE QUADRO CORREA,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720,4314407.0,200,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236,adrieu-cpinheiro@estudante.rs.gov.br,02046204069,FERNANDA DE QUADRO CORREA,1985-07-03,RESPONSAVEL


In [57]:
# pega cd_ibge de 7 dígitos do idt
query_idt = f"""
SELECT UNIQUE IAT.NRO_INT_ALUNO_ESTADO, IM.CD_IBGE_7 
FROM PDP_SE_STG.ISE_ALUNO_TURMA IAT
INNER JOIN PDP_SE_STG.ISE_TURMA T ON T.NRO_INT_TURMA = IAT.NRO_INT_TURMA
INNER JOIN PDP_SE_STG.ISE_CALENDARIO_ESTAB CE ON CE.NRO_INT_CALEND_ESTAB = IAT.NRO_INT_CALEND_ESTAB AND CURRENT DATE + 5 BETWEEN CE.DT_INICIO_ATIV AND CE.DT_FIM_ATIV
INNER JOIN PDP_SE_STG.ISE_ESTABELECIMENTO IE ON IAT.IDT_ESTAB = IE.IDT_ESTAB AND IE.IN_ESC_PRISIONAL <> 'S' AND IE.NRO_INT_DESIGNACAO NOT IN (10, 489)
INNER JOIN PDP_SE_STG.ISE_MUNICIPIO IM ON IE.NRO_INT_MUNICIPIO = IM.NRO_INT_MUNICIPIO AND IM.CD_IBGE_7 LIKE '43%'
INNER JOIN PDP_SE_STG.ISE_ALUNO IA ON IAT.NRO_INT_ALUNO_ESTADO = IA.NRO_INT_ALUNO_ESTADO
WHERE IAT.NRO_INT_SITUACAO IN (1,8)
AND ( 
	(T.CD_TIPO_ENSINO IN ('I2', 'E2', 'R2') -- SE FOR EM
	AND (INTEGER((DAYS(CURRENT DATE) - DAYS(IA.DT_NASCIMENTO)) / 365.25) <= 21)) -- TEM QUE TER 21 ANOS OU MENOS
	OR 
	(T.CD_TIPO_ENSINO IN ('S2', 'E9') -- SE FOR EJA
	AND (INTEGER((DAYS(CURRENT DATE) - DAYS(IA.DT_NASCIMENTO)) / 365.25) <= 29)) -- TEM QUE TER 29 ANOS OU MENOS
	)
"""

idts = consulta_datalake(query_idt)
idts = idts.drop_duplicates(subset='nro_int_aluno_estado').rename(columns={'cd_ibge_7': 'SETOR'})

rf = pd.merge(rf, idts, how='left')
rf

,nro_int_aluno_estado,COD_CPF,nm_aluno_puro,NOME_MAE,DTA_NASC,BAIRRO,NUM_END,CEP,CIDADE,VLR_RENDA_MEDIA,RG,COD_NIS,LOGRADOURO,UF,TELEFONE1,TELEFONE2,COMPLEMENTO,email_educar,CPF_RESP,NOME_RESP,DTA_NASC_RESP,ORIGEM_CPF,SETOR
0,6176291,05896382030,DANIELI VITORIA MARTINS GUINDANI,ELISANGELA RIBEIRO MARTINS,2008-06-26,PANORAMICO,30.0,95250000,4300802.0,300,00000000007136979395,2.375785e+10,TRAVESSA ARGENTINA,RS,54996024941,54997091641,,danieli-vmguindani@estudante.rs.gov.br,02438125071,ELISANGELA RIBEIRO MARTINS,1991-09-05,ALUNO,4300802
1,6176622,04757888007,SANDRA CRISTINA FERSTER,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,DISTRITO DE VILA ITAUBA,9.0,96990000,4307815.0,177,4757888007.0,2.360532e+10,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,RS,51997010479,5136167088,SN SN,sandra-cferster@educar.rs.gov.br,96609524068,IRACEMA LUIZA WAPPLER FERSTER,1981-08-30,ALUNO,4307815
2,6176620,04378585022,DAVI LUCAS BROLL RECH,GRAZIELA BROLL RECH,2009-10-15,SAO LUIS,9.0,96990000,4307815.0,163,1139758385,2.124554e+10,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,RS,51995938496,,SN CASA,davi-lbrech@educar.rs.gov.br,00952178001,GRAZIELA BROLL RECH,1986-07-14,ALUNO,4307815
3,6176621,05457643000,ILDOBRAND HUGO DA ROSA,ENIA EBERT DA ROSA,2010-01-04,SAO LUIZ,9.0,96990000,4307815.0,291,00000000003132521281,2.136268e+10,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,RS,,,SN CASA,ildobrand-hdrosa@educar.rs.gov.br,72177900000,ENIA EBERT DA ROSA,1970-09-29,ALUNO,4307815
4,6176623,05801189084,TALITA VITORIA DA SILVA,SIMONE LIRA DA SILVA,2009-08-18,LINHA DALCIN,9.0,96990000,4307815.0,177,5137047601.0,2.360445e+10,ESTRADA AGRICULTOR LINHA DALCIN,RS,51996564306,,SN SN,talita-vdsilva@educar.rs.gov.br,01970451033,SIMONE LIRA DA SILVA,1989-04-17,ALUNO,4307815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,4930127,00707337089,PATRICIA TEREZA DA CONCEICAO,SANTA RITA MARQUES TEREZA,1984-06-26,DOM ARMANDO,107.0,96745000,4305355.0,510,5088630701,2.099327e+10,RUA OTAVIO SOARES DA ROCHA,RS,51998143862,,,emilli-dchamorra@estudante.rs.gov.br,00707337089,PATRICIA TEREZA DA CONCEICAO,1984-06-26,RESPONSAVEL,4320008
101854,4881463,03526265003,LUANA STEFANI DOS SANTOS,ANGELA INES DOS SANTOS,1994-02-25,RENASCENCA,345.0,96815582,4316808.0,100,1118775749.0,1.640661e+10,RUA VEREADOR IVO CLAUDIO WEIGEL,RS,51920044923,,,pedro-hdsdsilva5@estudante.rs.gov.br,03526265003,LUANA STEFANI DOS SANTOS,1994-02-25,RESPONSAVEL,4316808
101855,5293295,01763855090,IZIELE REZENDES DE BORTOLI,IARA RAMOS REZENDES,1988-04-26,VARZEA,440.0,97650000,4310603.0,0,3101467508,1.297401e+10,RUA VINTE DE SETEMBRO,RS,55996770844,,FUNDOS,sofia-5293295@estudante.rs.gov.br,01763855090,IZIELE REZENDES DE BORTOLI,1988-04-26,RESPONSAVEL,4316907
101856,6102081,02046204069,FERNANDA DE QUADRO CORREA,MARIA HELENA DE QUADRO CORREA,1985-07-03,FRAGATA,790.0,96040720,4314407.0,200,00000000006104971351,1.632760e+10,RUA HENRIQUE DIAS,RS,53984847263,,BL 9 APT 236,adrieu-cpinheiro@estudante.rs.gov.br,02046204069,FERNANDA DE QUADRO CORREA,1985-07-03,RESPONSAVEL,4301305


In [58]:
def formata_nome(nome):
    try:
        nomes = []
        nomes = nome.split(' ')
        if len(nomes) >= 2:
            return nomes[0] + ' ' + nomes[-1]
        else:
            return nomes[0]
    except:
        return ''

rf['NOME CARTÃO'] = rf['nm_aluno_puro'].apply(formata_nome)
rf = rf.rename(columns={'nm_municipio': 'CIDADE','nm_aluno_puro': 'NOME COMPLETO', 
                        'email_educar': 'EMAIL', 'nro_int_aluno_estado': 'MATRICULA',
                        'NUM_END': 'NUMERO', 'UF': 'ESTADO(UF)', 'VLR_RENDA_MEDIA': 'RENDA_MEDIA'})
rf = rf[['NOME COMPLETO', 'NOME CARTÃO', 'COD_CPF', 'ORIGEM_CPF', 'MATRICULA', 'SETOR', 'NOME_MAE', 'DTA_NASC', 'LOGRADOURO', 'NUMERO', 'COMPLEMENTO', 'BAIRRO',
         'CIDADE', 'ESTADO(UF)', 'CEP', 'TELEFONE1', 'TELEFONE2', 'EMAIL', 'RENDA_MEDIA', 'NOME_RESP', 'CPF_RESP', 'DTA_NASC_RESP', 'COD_NIS', 'RG']]
rf

,NOME COMPLETO,NOME CARTÃO,COD_CPF,ORIGEM_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG
0,DANIELI VITORIA MARTINS GUINDANI,DANIELI GUINDANI,05896382030,ALUNO,6176291,4300802,ELISANGELA RIBEIRO MARTINS,2008-06-26,TRAVESSA ARGENTINA,30.0,,PANORAMICO,4300802.0,RS,95250000,54996024941,54997091641,danieli-vmguindani@estudante.rs.gov.br,300,ELISANGELA RIBEIRO MARTINS,02438125071,1991-09-05,2.375785e+10,00000000007136979395
1,SANDRA CRISTINA FERSTER,SANDRA FERSTER,04757888007,ALUNO,6176622,4307815,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,9.0,SN SN,DISTRITO DE VILA ITAUBA,4307815.0,RS,96990000,51997010479,5136167088,sandra-cferster@educar.rs.gov.br,177,IRACEMA LUIZA WAPPLER FERSTER,96609524068,1981-08-30,2.360532e+10,4757888007.0
2,DAVI LUCAS BROLL RECH,DAVI RECH,04378585022,ALUNO,6176620,4307815,GRAZIELA BROLL RECH,2009-10-15,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,9.0,SN CASA,SAO LUIS,4307815.0,RS,96990000,51995938496,,davi-lbrech@educar.rs.gov.br,163,GRAZIELA BROLL RECH,00952178001,1986-07-14,2.124554e+10,1139758385
3,ILDOBRAND HUGO DA ROSA,ILDOBRAND ROSA,05457643000,ALUNO,6176621,4307815,ENIA EBERT DA ROSA,2010-01-04,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,9.0,SN CASA,SAO LUIZ,4307815.0,RS,96990000,,,ildobrand-hdrosa@educar.rs.gov.br,291,ENIA EBERT DA ROSA,72177900000,1970-09-29,2.136268e+10,00000000003132521281
4,TALITA VITORIA DA SILVA,TALITA SILVA,05801189084,ALUNO,6176623,4307815,SIMONE LIRA DA SILVA,2009-08-18,ESTRADA AGRICULTOR LINHA DALCIN,9.0,SN SN,LINHA DALCIN,4307815.0,RS,96990000,51996564306,,talita-vdsilva@educar.rs.gov.br,177,SIMONE LIRA DA SILVA,01970451033,1989-04-17,2.360445e+10,5137047601.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101853,PATRICIA TEREZA DA CONCEICAO,PATRICIA CONCEICAO,00707337089,RESPONSAVEL,4930127,4320008,SANTA RITA MARQUES TEREZA,1984-06-26,RUA OTAVIO SOARES DA ROCHA,107.0,,DOM ARMANDO,4305355.0,RS,96745000,51998143862,,emilli-dchamorra@estudante.rs.gov.br,510,PATRICIA TEREZA DA CONCEICAO,00707337089,1984-06-26,2.099327e+10,5088630701
101854,LUANA STEFANI DOS SANTOS,LUANA SANTOS,03526265003,RESPONSAVEL,4881463,4316808,ANGELA INES DOS SANTOS,1994-02-25,RUA VEREADOR IVO CLAUDIO WEIGEL,345.0,,RENASCENCA,4316808.0,RS,96815582,51920044923,,pedro-hdsdsilva5@estudante.rs.gov.br,100,LUANA STEFANI DOS SANTOS,03526265003,1994-02-25,1.640661e+10,1118775749.0
101855,IZIELE REZENDES DE BORTOLI,IZIELE BORTOLI,01763855090,RESPONSAVEL,5293295,4316907,IARA RAMOS REZENDES,1988-04-26,RUA VINTE DE SETEMBRO,440.0,FUNDOS,VARZEA,4310603.0,RS,97650000,55996770844,,sofia-5293295@estudante.rs.gov.br,0,IZIELE REZENDES DE BORTOLI,01763855090,1988-04-26,1.297401e+10,3101467508
101856,FERNANDA DE QUADRO CORREA,FERNANDA CORREA,02046204069,RESPONSAVEL,6102081,4301305,MARIA HELENA DE QUADRO CORREA,1985-07-03,RUA HENRIQUE DIAS,790.0,BL 9 APT 236,FRAGATA,4314407.0,RS,96040720,53984847263,,adrieu-cpinheiro@estudante.rs.gov.br,200,FERNANDA DE QUADRO CORREA,02046204069,1985-07-03,1.632760e+10,00000000006104971351


In [59]:
rf.isna().sum()

NOME COMPLETO        0
NOME CARTÃO          0
COD_CPF              0
ORIGEM_CPF           0
MATRICULA            0
SETOR                0
NOME_MAE             2
DTA_NASC             0
LOGRADOURO           0
NUMERO            7883
COMPLEMENTO          0
BAIRRO               0
CIDADE               0
ESTADO(UF)           0
CEP                  0
TELEFONE1            0
TELEFONE2            0
EMAIL               17
RENDA_MEDIA          0
NOME_RESP          522
CPF_RESP             0
DTA_NASC_RESP      522
COD_NIS             70
RG               44439
dtype: int64

# Travas e Ajustes

In [60]:
print(len(rf['COD_CPF'].unique()))
teste = rf.drop_duplicates()
print(len(teste['COD_CPF'].unique()))
aaa = teste[['MATRICULA', 'COD_CPF']].groupby(['MATRICULA']).count()
aaa[aaa['COD_CPF'] > 1]

101683
101683


,COD_CPF
MATRICULA,


In [61]:
rf.isna().sum()

NOME COMPLETO        0
NOME CARTÃO          0
COD_CPF              0
ORIGEM_CPF           0
MATRICULA            0
SETOR                0
NOME_MAE             2
DTA_NASC             0
LOGRADOURO           0
NUMERO            7883
COMPLEMENTO          0
BAIRRO               0
CIDADE               0
ESTADO(UF)           0
CEP                  0
TELEFONE1            0
TELEFONE2            0
EMAIL               17
RENDA_MEDIA          0
NOME_RESP          522
CPF_RESP             0
DTA_NASC_RESP      522
COD_NIS             70
RG               44439
dtype: int64

In [62]:
base_repetidos = rf[rf['ORIGEM_CPF'] == 'ALUNO']
base_repetidos = base_repetidos[['MATRICULA', 'COD_CPF']].groupby(['COD_CPF']).count()
repetidos = list(base_repetidos[base_repetidos['MATRICULA'] > 1].reset_index(drop=False)['COD_CPF'].unique())
print(len(repetidos))
repetidos

9


['03809400009',
 '04533985025',
 '04537542012',
 '04719889018',
 '05534960000',
 '05833902002',
 '06133903007',
 '06285247064',
 '06430986071']

In [63]:
rf_ok = rf[~rf['COD_CPF'].isin(repetidos)]
rf_repetidos = rf[rf['COD_CPF'].isin(repetidos)]
matriculas_repetidas = list(rf_repetidos['MATRICULA'].unique())
rf_repetidos

,NOME COMPLETO,NOME CARTÃO,COD_CPF,ORIGEM_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG
4848,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,ALUNO,2745041,4314902,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741.0,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-dtrindade@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,1.633960e+10,NaN
6988,MAISA DA SILVA ABREU,MAISA ABREU,04537542012,ALUNO,6270052,4301750,CELIA SOUZA DA SILVA,2010-03-20,RUA VISCONDE DOIS,104.0,CASA,CENTRO,4301750.0,RS,96735000,51995007604,,maisa-dsabreu@estudante.rs.gov.br,27,OLAVO ABREU,00223994022,1978-05-17,2.281666e+10,1127750386.0
7306,JENIFER DE ALMEIDA SARAIVA,JENIFER SARAIVA,06285247064,ALUNO,5471953,4301602,LUCIANA LOPES DE ALMEIDA,2003-08-10,RUA ANTONIO FLORES,838.0,CASA FUNDOS,DAER,4301602.0,RS,96412550,51995469148,,jenifer-dasaraiva@estudante.rs.gov.br,0,JENIFER DE ALMEIDA SARAIVA,06285247064,2003-08-10,2.388096e+10,NaN
25538,DERIK RAFAEL ASSMANN FRANCO DA SILVA,DERIK SILVA,05534960000,ALUNO,5372163,4300604,NATALIA MICHELE DA SILVA ASSMANN,2009-11-27,TRAVESSA UNIDOS,50.0,CASA 03,CASCATA GLORIA,4314902.0,RS,91712260,51986400218,,derik-5372163@estudante.rs.gov.br,12,NATALIA MICHELE DA SILVA ASSMANN ROSA,03523463025,1994-06-14,2.143045e+10,NaN
33531,MARILIA GABRIELA DE JESUS SOUTO,MARILIA SOUTO,03809400009,ALUNO,4892944,4322400,MARIELE OLIVEIRA DE JESUS,2008-09-24,RUA ALCIDES TREIN,652.0,,CIDADE NOVA,4322400.0,RS,97511430,55999830240,,marilia-gsouto@estudante.rs.gov.br,25,MARIELE OLIVEIRA DE JESUS,02978125055,1987-02-13,2.362938e+10,1121144479
51124,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,ALUNO,6593765,4301750,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,NaN,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,katrielle-6593765@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,1.652074e+10,NaN
53671,JULIA DO PRADO CORREA,JULIA CORREA,06133903007,ALUNO,5943961,4301750,DEBORA MARQUES DO PRADO,2008-03-25,ESTRADA LINHA ALFREDO SILVEIRA,2694.0,,INTERIOR,4301750.0,RS,96735000,51999074253,,julia-dpcorrea@estudante.rs.gov.br,212,DEBORA MARQUES DO PRADO,02796573079,1988-11-13,1.632816e+10,1138229354
79572,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,ALUNO,6933243,4301750,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637.0,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,eduarda-6933243@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,2.123965e+10,1140905744
85599,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,ALUNO,6275054,4301750,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637.0,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,renato-rkarpinski@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,2.123965e+10,1140905744
86452,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,ALUNO,3711031,4301750,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,NaN,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,samela-pdsouza@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,1.652074e+10,NaN


In [64]:
query_nome = f"""
SELECT NRO_INT_ALUNO_ESTADO, NM_ALUNO_PURO 
FROM PDP_SE_STG.ISE_ALUNO
WHERE NRO_INT_ALUNO_ESTADO IN {str(tuple(rf_repetidos['MATRICULA'].astype(str).unique())).replace("'", "")}
"""

nomes = consulta_datalake(query_nome)
nomes = nomes.dropna(subset='nro_int_aluno_estado').rename(columns={'nro_int_aluno_estado': 'MATRICULA', 'nm_aluno_puro': 'NOME COMPLETO'})
nomes['MATRICULA'] = nomes['MATRICULA'].astype("int64")

mat_removidas = rf_repetidos.copy()
rf_repetidos = pd.merge(rf_repetidos, nomes, how='inner')

if salva:
    mat_removidas = mat_removidas[~mat_removidas['MATRICULA'].isin(rf_repetidos['MATRICULA'].unique())]
    salva_com_auto_fit_colunas(mat_removidas, Arquivo_Final + f'{data} Matrículas Removidas CPF Duplicado e Nome Dif.xlsx', 'removidos')
matriculas_removidas = list(mat_removidas['MATRICULA'].unique())
print(len(matriculas_removidas))
print(str(matriculas_removidas))

rf_repetidos

8
[6275054, 3711031, 4964263, 4273595, 5943962, 5977519, 6118386, 6710879]


,NOME COMPLETO,NOME CARTÃO,COD_CPF,ORIGEM_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG
0,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,ALUNO,2745041,4314902,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741.0,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-dtrindade@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,1.633960e+10,NaN
1,MAISA DA SILVA ABREU,MAISA ABREU,04537542012,ALUNO,6270052,4301750,CELIA SOUZA DA SILVA,2010-03-20,RUA VISCONDE DOIS,104.0,CASA,CENTRO,4301750.0,RS,96735000,51995007604,,maisa-dsabreu@estudante.rs.gov.br,27,OLAVO ABREU,00223994022,1978-05-17,2.281666e+10,1127750386.0
2,JENIFER DE ALMEIDA SARAIVA,JENIFER SARAIVA,06285247064,ALUNO,5471953,4301602,LUCIANA LOPES DE ALMEIDA,2003-08-10,RUA ANTONIO FLORES,838.0,CASA FUNDOS,DAER,4301602.0,RS,96412550,51995469148,,jenifer-dasaraiva@estudante.rs.gov.br,0,JENIFER DE ALMEIDA SARAIVA,06285247064,2003-08-10,2.388096e+10,NaN
3,DERIK RAFAEL ASSMANN FRANCO DA SILVA,DERIK SILVA,05534960000,ALUNO,5372163,4300604,NATALIA MICHELE DA SILVA ASSMANN,2009-11-27,TRAVESSA UNIDOS,50.0,CASA 03,CASCATA GLORIA,4314902.0,RS,91712260,51986400218,,derik-5372163@estudante.rs.gov.br,12,NATALIA MICHELE DA SILVA ASSMANN ROSA,03523463025,1994-06-14,2.143045e+10,NaN
4,MARILIA GABRIELA DE JESUS SOUTO,MARILIA SOUTO,03809400009,ALUNO,4892944,4322400,MARIELE OLIVEIRA DE JESUS,2008-09-24,RUA ALCIDES TREIN,652.0,,CIDADE NOVA,4322400.0,RS,97511430,55999830240,,marilia-gsouto@estudante.rs.gov.br,25,MARIELE OLIVEIRA DE JESUS,02978125055,1987-02-13,2.362938e+10,1121144479
5,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,ALUNO,6593765,4301750,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,NaN,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,katrielle-6593765@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,1.652074e+10,NaN
6,JULIA DO PRADO CORREA,JULIA CORREA,06133903007,ALUNO,5943961,4301750,DEBORA MARQUES DO PRADO,2008-03-25,ESTRADA LINHA ALFREDO SILVEIRA,2694.0,,INTERIOR,4301750.0,RS,96735000,51999074253,,julia-dpcorrea@estudante.rs.gov.br,212,DEBORA MARQUES DO PRADO,02796573079,1988-11-13,1.632816e+10,1138229354
7,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,ALUNO,6933243,4301750,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637.0,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,eduarda-6933243@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,2.123965e+10,1140905744
8,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,ALUNO,6723826,4314902,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741.0,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-6723826@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,1.633960e+10,NaN
9,DERIK RAFAEL ASSMANN FRANCO DA SILVA,DERIK SILVA,05534960000,ALUNO,7043403,4314902,NATALIA MICHELE DA SILVA ASSMANN,2009-11-27,TRAVESSA UNIDOS,50.0,CASA 03,CASCATA GLORIA,4314902.0,RS,91712260,51986400218,,derik-7043403@estudante.rs.gov.br,12,NATALIA MICHELE DA SILVA ASSMANN ROSA,03523463025,1994-06-14,2.143045e+10,NaN


In [65]:
rf = pd.concat([rf_ok, rf_repetidos], axis=0)
rf

,NOME COMPLETO,NOME CARTÃO,COD_CPF,ORIGEM_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG
0,DANIELI VITORIA MARTINS GUINDANI,DANIELI GUINDANI,05896382030,ALUNO,6176291,4300802,ELISANGELA RIBEIRO MARTINS,2008-06-26,TRAVESSA ARGENTINA,30.0,,PANORAMICO,4300802.0,RS,95250000,54996024941,54997091641,danieli-vmguindani@estudante.rs.gov.br,300,ELISANGELA RIBEIRO MARTINS,02438125071,1991-09-05,2.375785e+10,00000000007136979395
1,SANDRA CRISTINA FERSTER,SANDRA FERSTER,04757888007,ALUNO,6176622,4307815,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,9.0,SN SN,DISTRITO DE VILA ITAUBA,4307815.0,RS,96990000,51997010479,5136167088,sandra-cferster@educar.rs.gov.br,177,IRACEMA LUIZA WAPPLER FERSTER,96609524068,1981-08-30,2.360532e+10,4757888007.0
2,DAVI LUCAS BROLL RECH,DAVI RECH,04378585022,ALUNO,6176620,4307815,GRAZIELA BROLL RECH,2009-10-15,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,9.0,SN CASA,SAO LUIS,4307815.0,RS,96990000,51995938496,,davi-lbrech@educar.rs.gov.br,163,GRAZIELA BROLL RECH,00952178001,1986-07-14,2.124554e+10,1139758385
3,ILDOBRAND HUGO DA ROSA,ILDOBRAND ROSA,05457643000,ALUNO,6176621,4307815,ENIA EBERT DA ROSA,2010-01-04,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,9.0,SN CASA,SAO LUIZ,4307815.0,RS,96990000,,,ildobrand-hdrosa@educar.rs.gov.br,291,ENIA EBERT DA ROSA,72177900000,1970-09-29,2.136268e+10,00000000003132521281
4,TALITA VITORIA DA SILVA,TALITA SILVA,05801189084,ALUNO,6176623,4307815,SIMONE LIRA DA SILVA,2009-08-18,ESTRADA AGRICULTOR LINHA DALCIN,9.0,SN SN,LINHA DALCIN,4307815.0,RS,96990000,51996564306,,talita-vdsilva@educar.rs.gov.br,177,SIMONE LIRA DA SILVA,01970451033,1989-04-17,2.360445e+10,5137047601.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,ALUNO,6593765,4301750,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,NaN,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,katrielle-6593765@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,1.652074e+10,NaN
6,JULIA DO PRADO CORREA,JULIA CORREA,06133903007,ALUNO,5943961,4301750,DEBORA MARQUES DO PRADO,2008-03-25,ESTRADA LINHA ALFREDO SILVEIRA,2694.0,,INTERIOR,4301750.0,RS,96735000,51999074253,,julia-dpcorrea@estudante.rs.gov.br,212,DEBORA MARQUES DO PRADO,02796573079,1988-11-13,1.632816e+10,1138229354
7,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,ALUNO,6933243,4301750,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637.0,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,eduarda-6933243@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,2.123965e+10,1140905744
8,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,ALUNO,6723826,4314902,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741.0,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-6723826@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,1.633960e+10,NaN


In [66]:
final = rf.copy()
len(final['MATRICULA'].unique())

101850

**1)** CPF_RESP em branco devem ser eliminados se menor de 18 anos e separados p/ação corretiva posterior

In [67]:
from datetime import date, datetime

dia, mes = data.split('-')
dia = int(dia)
mes = int(mes)

def calcula_idade(dta_nasc):
    dta_nasc = datetime.strptime(dta_nasc, '%Y-%m-%d').date()
    hoje = date(2025, mes, dia)
    
    return hoje.year - dta_nasc.year - ((hoje.month, hoje.day) < (dta_nasc.month, dta_nasc.day))

final['IDADE'] = final['DTA_NASC']
final['IDADE'] = final['IDADE'].apply(calcula_idade)
final

,NOME COMPLETO,NOME CARTÃO,COD_CPF,ORIGEM_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG,IDADE
0,DANIELI VITORIA MARTINS GUINDANI,DANIELI GUINDANI,05896382030,ALUNO,6176291,4300802,ELISANGELA RIBEIRO MARTINS,2008-06-26,TRAVESSA ARGENTINA,30.0,,PANORAMICO,4300802.0,RS,95250000,54996024941,54997091641,danieli-vmguindani@estudante.rs.gov.br,300,ELISANGELA RIBEIRO MARTINS,02438125071,1991-09-05,2.375785e+10,00000000007136979395,17
1,SANDRA CRISTINA FERSTER,SANDRA FERSTER,04757888007,ALUNO,6176622,4307815,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,9.0,SN SN,DISTRITO DE VILA ITAUBA,4307815.0,RS,96990000,51997010479,5136167088,sandra-cferster@educar.rs.gov.br,177,IRACEMA LUIZA WAPPLER FERSTER,96609524068,1981-08-30,2.360532e+10,4757888007.0,15
2,DAVI LUCAS BROLL RECH,DAVI RECH,04378585022,ALUNO,6176620,4307815,GRAZIELA BROLL RECH,2009-10-15,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,9.0,SN CASA,SAO LUIS,4307815.0,RS,96990000,51995938496,,davi-lbrech@educar.rs.gov.br,163,GRAZIELA BROLL RECH,00952178001,1986-07-14,2.124554e+10,1139758385,15
3,ILDOBRAND HUGO DA ROSA,ILDOBRAND ROSA,05457643000,ALUNO,6176621,4307815,ENIA EBERT DA ROSA,2010-01-04,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,9.0,SN CASA,SAO LUIZ,4307815.0,RS,96990000,,,ildobrand-hdrosa@educar.rs.gov.br,291,ENIA EBERT DA ROSA,72177900000,1970-09-29,2.136268e+10,00000000003132521281,15
4,TALITA VITORIA DA SILVA,TALITA SILVA,05801189084,ALUNO,6176623,4307815,SIMONE LIRA DA SILVA,2009-08-18,ESTRADA AGRICULTOR LINHA DALCIN,9.0,SN SN,LINHA DALCIN,4307815.0,RS,96990000,51996564306,,talita-vdsilva@educar.rs.gov.br,177,SIMONE LIRA DA SILVA,01970451033,1989-04-17,2.360445e+10,5137047601.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,ALUNO,6593765,4301750,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,NaN,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,katrielle-6593765@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,1.652074e+10,NaN,17
6,JULIA DO PRADO CORREA,JULIA CORREA,06133903007,ALUNO,5943961,4301750,DEBORA MARQUES DO PRADO,2008-03-25,ESTRADA LINHA ALFREDO SILVEIRA,2694.0,,INTERIOR,4301750.0,RS,96735000,51999074253,,julia-dpcorrea@estudante.rs.gov.br,212,DEBORA MARQUES DO PRADO,02796573079,1988-11-13,1.632816e+10,1138229354,17
7,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,ALUNO,6933243,4301750,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637.0,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,eduarda-6933243@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,2.123965e+10,1140905744,16
8,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,ALUNO,6723826,4314902,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741.0,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-6723826@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,1.633960e+10,NaN,17


In [68]:
'''maiores = final[final['IDADE'] >= 18]
menores = final[final['IDADE'] < 18]

acao_corretiva_posterior = menores[menores['CPF_RESP'] == '']
if salva:
    print(acao_corretiva_posterior['MATRICULA'].nunique())
    salva_com_auto_fit_colunas(acao_corretiva_posterior, Arquivo_Final + f'{data} Menores sem Resp TJE.xlsx', 'menores')
menores_ok = menores[menores['CPF_RESP'] != '']

final = pd.concat([maiores, menores_ok], axis=0)
final'''
print('não precisa mais de menores ok, cras e banri deram ok 21-02-25')

não precisa mais de menores ok, cras e banri deram ok 21-02-25


In [69]:
'''str(list(acao_corretiva_posterior['MATRICULA'].unique()))'''
print('não precisa mais de menores ok, cras e banri deram ok 21-02-25')

não precisa mais de menores ok, cras e banri deram ok 21-02-25


**2)** SETOR em branco deve ser preenchido com CIDADE

In [70]:
setor_ok = final[(final['SETOR'] != '') & (~final['SETOR'].isna())]
setor_em_branco = final[(final['SETOR'] == '') | (final['SETOR'].isna())]
setor_em_branco['SETOR'] = setor_em_branco['CIDADE']

final = pd.concat([setor_ok, setor_em_branco], axis=0)
final

,NOME COMPLETO,NOME CARTÃO,COD_CPF,ORIGEM_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG,IDADE
0,DANIELI VITORIA MARTINS GUINDANI,DANIELI GUINDANI,05896382030,ALUNO,6176291,4300802.0,ELISANGELA RIBEIRO MARTINS,2008-06-26,TRAVESSA ARGENTINA,30.0,,PANORAMICO,4300802.0,RS,95250000,54996024941,54997091641,danieli-vmguindani@estudante.rs.gov.br,300,ELISANGELA RIBEIRO MARTINS,02438125071,1991-09-05,2.375785e+10,00000000007136979395,17
1,SANDRA CRISTINA FERSTER,SANDRA FERSTER,04757888007,ALUNO,6176622,4307815.0,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,9.0,SN SN,DISTRITO DE VILA ITAUBA,4307815.0,RS,96990000,51997010479,5136167088,sandra-cferster@educar.rs.gov.br,177,IRACEMA LUIZA WAPPLER FERSTER,96609524068,1981-08-30,2.360532e+10,4757888007.0,15
2,DAVI LUCAS BROLL RECH,DAVI RECH,04378585022,ALUNO,6176620,4307815.0,GRAZIELA BROLL RECH,2009-10-15,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,9.0,SN CASA,SAO LUIS,4307815.0,RS,96990000,51995938496,,davi-lbrech@educar.rs.gov.br,163,GRAZIELA BROLL RECH,00952178001,1986-07-14,2.124554e+10,1139758385,15
3,ILDOBRAND HUGO DA ROSA,ILDOBRAND ROSA,05457643000,ALUNO,6176621,4307815.0,ENIA EBERT DA ROSA,2010-01-04,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,9.0,SN CASA,SAO LUIZ,4307815.0,RS,96990000,,,ildobrand-hdrosa@educar.rs.gov.br,291,ENIA EBERT DA ROSA,72177900000,1970-09-29,2.136268e+10,00000000003132521281,15
4,TALITA VITORIA DA SILVA,TALITA SILVA,05801189084,ALUNO,6176623,4307815.0,SIMONE LIRA DA SILVA,2009-08-18,ESTRADA AGRICULTOR LINHA DALCIN,9.0,SN SN,LINHA DALCIN,4307815.0,RS,96990000,51996564306,,talita-vdsilva@educar.rs.gov.br,177,SIMONE LIRA DA SILVA,01970451033,1989-04-17,2.360445e+10,5137047601.0,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,ALUNO,6593765,4301750.0,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,NaN,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,katrielle-6593765@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,1.652074e+10,NaN,17
6,JULIA DO PRADO CORREA,JULIA CORREA,06133903007,ALUNO,5943961,4301750.0,DEBORA MARQUES DO PRADO,2008-03-25,ESTRADA LINHA ALFREDO SILVEIRA,2694.0,,INTERIOR,4301750.0,RS,96735000,51999074253,,julia-dpcorrea@estudante.rs.gov.br,212,DEBORA MARQUES DO PRADO,02796573079,1988-11-13,1.632816e+10,1138229354,17
7,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,ALUNO,6933243,4301750.0,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637.0,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,eduarda-6933243@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,2.123965e+10,1140905744,16
8,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,ALUNO,6723826,4314902.0,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741.0,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-6723826@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,1.633960e+10,NaN,17


**3)** Qdo maior de 18 anos, limpar os campos *_RESP **removido 18/03!**

In [71]:
def maior_idade(idade):
    if idade >= 18:
        return 1
    else:
        return 0

'''
maiores = final[final['IDADE'] >= 18]
menores = final[final['IDADE'] < 18]

maiores['NOME_RESP'] = ''
maiores['CPF_RESP'] = ''
maiores['DTA_NASC_RESP'] = ''

final = pd.concat([maiores, menores], axis=0)
'''
final = final.rename(columns={'IDADE': 'MAIOR DE IDADE'})
final['MAIOR DE IDADE'] = final['MAIOR DE IDADE'].apply(maior_idade)
final

,NOME COMPLETO,NOME CARTÃO,COD_CPF,ORIGEM_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG,MAIOR DE IDADE
0,DANIELI VITORIA MARTINS GUINDANI,DANIELI GUINDANI,05896382030,ALUNO,6176291,4300802.0,ELISANGELA RIBEIRO MARTINS,2008-06-26,TRAVESSA ARGENTINA,30.0,,PANORAMICO,4300802.0,RS,95250000,54996024941,54997091641,danieli-vmguindani@estudante.rs.gov.br,300,ELISANGELA RIBEIRO MARTINS,02438125071,1991-09-05,2.375785e+10,00000000007136979395,0
1,SANDRA CRISTINA FERSTER,SANDRA FERSTER,04757888007,ALUNO,6176622,4307815.0,IRACEMA LUIZA WAPPLER FERSTER,2010-04-20,ESTRADA AGRICULTOR SANTA TEREZINHA INTERIOR,9.0,SN SN,DISTRITO DE VILA ITAUBA,4307815.0,RS,96990000,51997010479,5136167088,sandra-cferster@educar.rs.gov.br,177,IRACEMA LUIZA WAPPLER FERSTER,96609524068,1981-08-30,2.360532e+10,4757888007.0,0
2,DAVI LUCAS BROLL RECH,DAVI RECH,04378585022,ALUNO,6176620,4307815.0,GRAZIELA BROLL RECH,2009-10-15,ESTRADA SAO LUIS DISTRITO DE ESTRELA VELHA,9.0,SN CASA,SAO LUIS,4307815.0,RS,96990000,51995938496,,davi-lbrech@educar.rs.gov.br,163,GRAZIELA BROLL RECH,00952178001,1986-07-14,2.124554e+10,1139758385,0
3,ILDOBRAND HUGO DA ROSA,ILDOBRAND ROSA,05457643000,ALUNO,6176621,4307815.0,ENIA EBERT DA ROSA,2010-01-04,ESTRADA SAO LUIZ INTEIOR DE ESTRELA VELHA,9.0,SN CASA,SAO LUIZ,4307815.0,RS,96990000,,,ildobrand-hdrosa@educar.rs.gov.br,291,ENIA EBERT DA ROSA,72177900000,1970-09-29,2.136268e+10,00000000003132521281,0
4,TALITA VITORIA DA SILVA,TALITA SILVA,05801189084,ALUNO,6176623,4307815.0,SIMONE LIRA DA SILVA,2009-08-18,ESTRADA AGRICULTOR LINHA DALCIN,9.0,SN SN,LINHA DALCIN,4307815.0,RS,96990000,51996564306,,talita-vdsilva@educar.rs.gov.br,177,SIMONE LIRA DA SILVA,01970451033,1989-04-17,2.360445e+10,5137047601.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,ALUNO,6593765,4301750.0,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,NaN,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,katrielle-6593765@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,1.652074e+10,NaN,0
6,JULIA DO PRADO CORREA,JULIA CORREA,06133903007,ALUNO,5943961,4301750.0,DEBORA MARQUES DO PRADO,2008-03-25,ESTRADA LINHA ALFREDO SILVEIRA,2694.0,,INTERIOR,4301750.0,RS,96735000,51999074253,,julia-dpcorrea@estudante.rs.gov.br,212,DEBORA MARQUES DO PRADO,02796573079,1988-11-13,1.632816e+10,1138229354,0
7,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,ALUNO,6933243,4301750.0,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637.0,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,eduarda-6933243@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,2.123965e+10,1140905744,0
8,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,ALUNO,6723826,4314902.0,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741.0,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-6723826@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,1.633960e+10,NaN,0


**4)** Verifica se primeiro nome do ISE e CAD batem

In [72]:
'''
query_nome = f"""
SELECT NRO_INT_ALUNO_ESTADO, NM_ALUNO_PURO 
FROM PDP_SE_STG.ISE_ALUNO
"""

nomes = consulta_datalake(query_nome)
nomes = nomes.dropna(subset='nro_int_aluno_estado').rename(columns={'nro_int_aluno_estado': 'MATRICULA', 'nm_aluno_puro': 'NOME COMPLETO'})
nomes['MATRICULA'] = nomes['MATRICULA'].astype("int64")

rf_repetidos = pd.merge(rf_repetidos, nomes, how='inner')
rf_repetidos
'''

print('to do')

to do


**Novo layout ORIGEM_CPF**: Indica se o COD_CPF se refere ao do estudante (ALUNO), ao do responsável (RESPONSAVEL) ou se o próprio estudante é o responsável familiar (ALUNO/RESPONSAVEL) , prioridade em pegar do aluno, só pegar responsável se não achar do estudante

In [73]:
final = final.rename(columns={'ORIGEM_CPF': 'OLD_ORIGEM_CPF'})
final['ORIGEM_CPF'] = final['OLD_ORIGEM_CPF'].replace(['ALUNO/RESPONSAVEL'], ['RESPONSAVEL'])

origem_aluno = final[final['OLD_ORIGEM_CPF'] == 'ALUNO']
origem_resp = final[final['OLD_ORIGEM_CPF'] != 'ALUNO']

origem_aluno_resp = origem_aluno[origem_aluno['COD_CPF'] == origem_aluno['CPF_RESP']]
origem_aluno_aluno = origem_aluno[origem_aluno['COD_CPF'] != origem_aluno['CPF_RESP']]

origem_aluno_resp['ORIGEM_CPF'] = 'ALUNO/RESPONSAVEL'

final = pd.concat([origem_resp, origem_aluno_resp, origem_aluno_aluno], axis=0).drop(columns='OLD_ORIGEM_CPF')
final['SETOR'] = final['SETOR'].astype("int64")

final['NUMERO'] = final['NUMERO'].astype(str)
final['NUMERO'] = final['NUMERO'].str.replace('nan', 'S/N')
final['NUMERO'] = final['NUMERO'].str.replace('.0', '')

final['COD_NIS'] = final['COD_NIS'].astype(str)
final['COD_NIS'] = final['COD_NIS'].str.replace('nan', 'S/N')
final['COD_NIS'] = final['COD_NIS'].str.replace('.0', '')

final['RG'] = final['RG'].fillna('S/N').str.replace('.0', '')
final['NOME_MAE'] = final['NOME_MAE'].fillna('Genitora Desconhecida')
final

,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG,MAIOR DE IDADE,ORIGEM_CPF
8167,TALITA GARCIA MEIRELES,TALITA MEIRELES,04009923024,2992058,4304606,SIMONE GARCIA MEIRELES,1997-01-22,RUA PITANGUEIRAS,117,,CENTRO,4313375.0,RS,92480000,51998523865,,talita-2992058@estudante.rs.gov.br,0,TALITA GARCIA MEIRELES,04009923024,1997-01-22,20768089691,00000000001122916305,1,RESPONSAVEL
12567,BRENDA CAROLINA MAIDANA DA SILVA,BRENDA SILVA,04054479065,187776,4322509,CARLA MAIDANA,1996-04-07,RUA DARI TISSOTI,241,,THOME DE SOUZA,4310207.0,RS,98700000,54991265029,,brenda-187776@estudante.rs.gov.br,0,BRENDA CAROLINA MAIDANA DA SILVA,04054479065,1996-04-07,20670488415,1119117461,1,RESPONSAVEL
13059,SIMONE DA SILVA MENDES,SIMONE MENDES,05137976024,5371242,4300406,SANDRA AMALIA LOPES CARVALHO,2001-02-01,RUA LIBERATO SALZANO,55,FUNDOS CASA 2,SAO CRISTOVAO,4314100.0,RS,99064050,55999101514,,simone-5371242@estudante.rs.gov.br,75,SIMONE DA SILVA MENDES,05137976024,2001-02-01,16629434553,8115063946,1,RESPONSAVEL
16398,GRACIELE LEAL RAMOS,GRACIELE RAMOS,04352845094,2886708,4316402,MARIA CRISTINA LEAL RAMOS,1996-08-01,ESTRADA RINCAO DOS NEGROS,S/N,SN CASA,PICADAS,4316402.0,RS,97590000,55999573097,,graciele-lramos@estudante.rs.gov.br,444,GRACIELE LEAL RAMOS,04352845094,1996-08-01,21056093392,8126130833,1,RESPONSAVEL
25347,AMANDA DOS SANTOS AJALA,AMANDA AJALA,04492904093,662091,4316402,GLECI CAMARGO DOS SANTOS,2002-03-18,RUA CACEQUI,1199,CASA,LAGOA,4316402.0,RS,97590000,55998424663,,amanda-662091@estudante.rs.gov.br,50,AMANDA DOS SANTOS AJALA,04492904093,2002-03-18,21215081156,00000000007127394661,1,RESPONSAVEL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,KATRIELLE SILVA DA SILVEIRA,KATRIELLE SILVEIRA,06430986071,6593765,4301750,ANA PAULA OLIVEIRA DA SILVA,2007-11-15,ESTRADA PALMEIRA,S/N,SN,INTERIOR,4301750.0,RS,96735000,51998931945,,katrielle-6593765@estudante.rs.gov.br,150,ANA PAULA OLIVEIRA DA SILVA,03329830050,1993-09-07,16520739082,S/N,0,ALUNO
6,JULIA DO PRADO CORREA,JULIA CORREA,06133903007,5943961,4301750,DEBORA MARQUES DO PRADO,2008-03-25,ESTRADA LINHA ALFREDO SILVEIRA,2694,,INTERIOR,4301750.0,RS,96735000,51999074253,,julia-dpcorrea@estudante.rs.gov.br,212,DEBORA MARQUES DO PRADO,02796573079,1988-11-13,16328158654,1138229354,0,ALUNO
7,EDUARDA OLIVEIRA DA SILVA,EDUARDA SILVA,04533985025,6933243,4301750,SILVANA DE ABREU OLIVEIRA,2009-04-23,ESTRADA DO HERVAL,4637,CASA,INTERIOR,4301750.0,RS,96735000,51980570597,,eduarda-6933243@estudante.rs.gov.br,0,SILVANA DE ABREU OLIVEIRA,01093201070,1985-03-03,21239645114,1140905744,0,ALUNO
8,YASMYM DA COSTA TRINDADE,YASMYM TRINDADE,04719889018,6723826,4314902,DANIELI LEOPOLDO DA COSTA,2008-01-03,ACESSO QUINZE,5741,CASA 03 CASA 164,RESTINGA,4314902.0,RS,91790000,5192460691,,yasmym-6723826@estudante.rs.gov.br,0,DANIELI LEOPOLDO DA COSTA,84966254000,1993-09-18,16339600604,S/N,0,ALUNO


**mudanças 11/04:**

1) nome da coluna COD_CPF_RESP para CPF_RESP;
2) ta vindo em branco algumas linhas da coluna do RESP, provavelmente pela regra do maior de idade; nesses casos, coloca o da própria pessoa (CPF_RESP fica igual ao COD_CPF, DT_NASC_RESP fica igual a DT_NASC e NOME_RESP fica igual ao NOME_BENEFICIARIO)
3) precisamos do arquivo em csv, pode gerar um adicional .csv

In [74]:
# divide em campos onde existem rf=1 e onde não existem
final_ok = final[final['CPF_RESP'] != '']
final_not_ok = final[final['CPF_RESP'] == '']
print(str(final_ok.shape[0]) + ' matrículas possuem RF=1, enquanto que ' + str(final_not_ok.shape[0]) + ' não possuem')

101328 matrículas possuem RF=1, enquanto que 522 não possuem


1) Qdo ORIGEM_CPF <> ALUNO, usar os próprios campos COD_CPF, NOME_BENEFICIARIO e DT_NASC_BENEFICIARIO;

In [75]:
origem_not_aluno = final_not_ok[final_not_ok['ORIGEM_CPF'] != 'ALUNO']
origem_not_aluno['CPF_RESP'] = origem_not_aluno['COD_CPF']
origem_not_aluno['DTA_NASC_RESP'] = origem_not_aluno['DTA_NASC']
origem_not_aluno['NOME_RESP'] = origem_not_aluno['NOME COMPLETO']
print(str(origem_not_aluno.shape[0]) + ' alunos sem RF=1 possuem origem <> ALUNO')

19 alunos sem RF=1 possuem origem <> ALUNO


2) Qdo ORIGEM_CPF = ALUNO, pegar o menor RF acima de 18 anos

In [76]:
origem_aluno = final_not_ok[final_not_ok['ORIGEM_CPF'] == 'ALUNO']
origem_aluno['COD_CPF'] = origem_aluno['COD_CPF'].astype("int64")

# obtém lista de cod_fam dos alunos em origem_aluno
lista_cod_fam = df_cad[df_cad['COD_CPF'].isin(origem_aluno['COD_CPF'].unique())]['COD_FAM'].unique()

# relação entre cpf e cod_fam
cpf_cod_fam = df_cad[df_cad['COD_CPF'].isin(origem_aluno['COD_CPF'].unique())][['COD_CPF', 'COD_FAM']]

# menor rf de cada cod_fam
menor_rf = df_cad[df_cad['COD_FAM'].isin(lista_cod_fam)].dropna(subset='COD_CPF')[['COD_CPF', 'NOME', 'DTA_NASC', 'COD_FAM', 'COD_PARENTESCO']]
menor_rf = menor_rf.sort_values(by='COD_PARENTESCO', ascending=True).drop_duplicates(subset='COD_FAM', keep='first')
menor_rf = menor_rf.rename(columns={'COD_CPF': 'CPF_RESP', 'NOME': 'NOME_RESP', 'DTA_NASC': 'DTA_NASC_RESP'})

origem_aluno = origem_aluno.drop(columns=['CPF_RESP', 'NOME_RESP', 'DTA_NASC_RESP'])
origem_aluno = pd.merge(origem_aluno, cpf_cod_fam)
origem_aluno = pd.merge(origem_aluno, menor_rf).drop(columns=['COD_FAM', 'COD_PARENTESCO'])

origem_aluno['COD_CPF'] = origem_aluno['COD_CPF'].astype("int64").astype(str)
origem_aluno['COD_CPF'] = origem_aluno['COD_CPF'].str.zfill(11)

origem_aluno['CPF_RESP'] = origem_aluno['CPF_RESP'].astype("int64").astype(str)
origem_aluno['CPF_RESP'] = origem_aluno['CPF_RESP'].str.zfill(11)
print(str(origem_aluno.shape[0]) + ' alunos sem RF=1 possuem origem = ALUNO')

503 alunos sem RF=1 possuem origem = ALUNO


In [77]:
final_not_ok = pd.concat([origem_not_aluno, origem_aluno])
final = pd.concat([final_ok, final_not_ok])
final

,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG,MAIOR DE IDADE,ORIGEM_CPF
8167,TALITA GARCIA MEIRELES,TALITA MEIRELES,04009923024,2992058,4304606,SIMONE GARCIA MEIRELES,1997-01-22,RUA PITANGUEIRAS,117,,CENTRO,4313375.0,RS,92480000,51998523865,,talita-2992058@estudante.rs.gov.br,0,TALITA GARCIA MEIRELES,04009923024,1997-01-22,20768089691,00000000001122916305,1,RESPONSAVEL
12567,BRENDA CAROLINA MAIDANA DA SILVA,BRENDA SILVA,04054479065,187776,4322509,CARLA MAIDANA,1996-04-07,RUA DARI TISSOTI,241,,THOME DE SOUZA,4310207.0,RS,98700000,54991265029,,brenda-187776@estudante.rs.gov.br,0,BRENDA CAROLINA MAIDANA DA SILVA,04054479065,1996-04-07,20670488415,1119117461,1,RESPONSAVEL
13059,SIMONE DA SILVA MENDES,SIMONE MENDES,05137976024,5371242,4300406,SANDRA AMALIA LOPES CARVALHO,2001-02-01,RUA LIBERATO SALZANO,55,FUNDOS CASA 2,SAO CRISTOVAO,4314100.0,RS,99064050,55999101514,,simone-5371242@estudante.rs.gov.br,75,SIMONE DA SILVA MENDES,05137976024,2001-02-01,16629434553,8115063946,1,RESPONSAVEL
16398,GRACIELE LEAL RAMOS,GRACIELE RAMOS,04352845094,2886708,4316402,MARIA CRISTINA LEAL RAMOS,1996-08-01,ESTRADA RINCAO DOS NEGROS,S/N,SN CASA,PICADAS,4316402.0,RS,97590000,55999573097,,graciele-lramos@estudante.rs.gov.br,444,GRACIELE LEAL RAMOS,04352845094,1996-08-01,21056093392,8126130833,1,RESPONSAVEL
25347,AMANDA DOS SANTOS AJALA,AMANDA AJALA,04492904093,662091,4316402,GLECI CAMARGO DOS SANTOS,2002-03-18,RUA CACEQUI,1199,CASA,LAGOA,4316402.0,RS,97590000,55998424663,,amanda-662091@estudante.rs.gov.br,50,AMANDA DOS SANTOS AJALA,04492904093,2002-03-18,21215081156,00000000007127394661,1,RESPONSAVEL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,MARIANE DE AVILA DE LIMA,MARIANE LIMA,05988221009,7050459,4304663,PRISCILA NEITZKE DE AVILA,2009-07-10,RUA LUIZ MARIA DA ROCHA,128,,JARDIM AMERICA,4304663.0,RS,96160000,53991735119,,mariane-7050459@estudante.rs.gov.br,0,MARIANE DE AVILA DE LIMA,05988221009,2009-07-10,23790521066,S/N,0,ALUNO
499,ELISIANE SANTOS CUNHA,ELISIANE CUNHA,07022746079,6957477,4303509,CARMEM LIOMAR DOS SANTOS E SANTOS,2007-10-10,RUA EDERALDO DE SOUZA GOMES,12,,GETLIO VARGAS,4303509.0,RS,96785000,51989305983,,elisiane-6957477@estudante.rs.gov.br,338,ELISIANE SANTOS CUNHA,07022746079,2007-10-10,21224567716,1137719281,0,ALUNO
500,GABRIEL MORAIS GONCALVES,GABRIEL GONCALVES,05531638094,6973808,4301305,ADRIANA FERNANDA DE MORAIS,2009-10-04,RUA JOAO INACIO SULZBACH,574,,INDUSTRIAS,4307807.0,RS,95880000,51997359224,,gabriel-6973808@estudante.rs.gov.br,100,GABRIEL MORAIS GONCALVES,05531638094,2009-10-04,23662803107,S/N,0,ALUNO
501,KALYEL DORNELLES BERKAIER,KALYEL BERKAIER,06125171019,6921837,4309209,CAMILA ANDREADORNELLES REHBLEIN,2009-05-20,RUA GOIAS,298,,FATIMA,4303103.0,RS,94955180,51985117948,,kalyel-6921837@estudante.rs.gov.br,0,KALYEL DORNELLES BERKAIER,06125171019,2009-05-20,23707907690,NaN,0,ALUNO


**Remoção de CPFs duplicados em COD_FAM diferentes:**

[18:31, 28/06/2024] Tales SEAS: Oi Guilherme! Tudo bem! Nesses casos de CPF duplicado, em está atribuído ao mesma pessoa da família, pode unificar, caso esteja atribuído a pessoas diferentes, nesses casos teríamos que verificar, pois pode ter ocorrido um erro no preenchimento ou na própria consolidação dos dados da família. Mas se está atribuído a mesma pessoa, pode unificar

[18:36, 28/06/2024] Guilherme Simionato: possui codigo familiar diferente

[18:37, 28/06/2024] Tales SEAS: Pode ser um caso em que a pessoa migrou de cadastro, foi transferida, mas não foi excluída da composição familiar em que estava.

[18:38, 28/06/2024] Tales SEAS: Nesse caso tu terias não poderia considerar, pode ser um problema

In [78]:
final = final.drop_duplicates(subset=['COD_CPF', 'MATRICULA'], keep=False)
final

,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE1,TELEFONE2,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP,COD_NIS,RG,MAIOR DE IDADE,ORIGEM_CPF
8167,TALITA GARCIA MEIRELES,TALITA MEIRELES,04009923024,2992058,4304606,SIMONE GARCIA MEIRELES,1997-01-22,RUA PITANGUEIRAS,117,,CENTRO,4313375.0,RS,92480000,51998523865,,talita-2992058@estudante.rs.gov.br,0,TALITA GARCIA MEIRELES,04009923024,1997-01-22,20768089691,00000000001122916305,1,RESPONSAVEL
12567,BRENDA CAROLINA MAIDANA DA SILVA,BRENDA SILVA,04054479065,187776,4322509,CARLA MAIDANA,1996-04-07,RUA DARI TISSOTI,241,,THOME DE SOUZA,4310207.0,RS,98700000,54991265029,,brenda-187776@estudante.rs.gov.br,0,BRENDA CAROLINA MAIDANA DA SILVA,04054479065,1996-04-07,20670488415,1119117461,1,RESPONSAVEL
13059,SIMONE DA SILVA MENDES,SIMONE MENDES,05137976024,5371242,4300406,SANDRA AMALIA LOPES CARVALHO,2001-02-01,RUA LIBERATO SALZANO,55,FUNDOS CASA 2,SAO CRISTOVAO,4314100.0,RS,99064050,55999101514,,simone-5371242@estudante.rs.gov.br,75,SIMONE DA SILVA MENDES,05137976024,2001-02-01,16629434553,8115063946,1,RESPONSAVEL
16398,GRACIELE LEAL RAMOS,GRACIELE RAMOS,04352845094,2886708,4316402,MARIA CRISTINA LEAL RAMOS,1996-08-01,ESTRADA RINCAO DOS NEGROS,S/N,SN CASA,PICADAS,4316402.0,RS,97590000,55999573097,,graciele-lramos@estudante.rs.gov.br,444,GRACIELE LEAL RAMOS,04352845094,1996-08-01,21056093392,8126130833,1,RESPONSAVEL
25347,AMANDA DOS SANTOS AJALA,AMANDA AJALA,04492904093,662091,4316402,GLECI CAMARGO DOS SANTOS,2002-03-18,RUA CACEQUI,1199,CASA,LAGOA,4316402.0,RS,97590000,55998424663,,amanda-662091@estudante.rs.gov.br,50,AMANDA DOS SANTOS AJALA,04492904093,2002-03-18,21215081156,00000000007127394661,1,RESPONSAVEL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,MARIANE DE AVILA DE LIMA,MARIANE LIMA,05988221009,7050459,4304663,PRISCILA NEITZKE DE AVILA,2009-07-10,RUA LUIZ MARIA DA ROCHA,128,,JARDIM AMERICA,4304663.0,RS,96160000,53991735119,,mariane-7050459@estudante.rs.gov.br,0,MARIANE DE AVILA DE LIMA,05988221009,2009-07-10,23790521066,S/N,0,ALUNO
499,ELISIANE SANTOS CUNHA,ELISIANE CUNHA,07022746079,6957477,4303509,CARMEM LIOMAR DOS SANTOS E SANTOS,2007-10-10,RUA EDERALDO DE SOUZA GOMES,12,,GETLIO VARGAS,4303509.0,RS,96785000,51989305983,,elisiane-6957477@estudante.rs.gov.br,338,ELISIANE SANTOS CUNHA,07022746079,2007-10-10,21224567716,1137719281,0,ALUNO
500,GABRIEL MORAIS GONCALVES,GABRIEL GONCALVES,05531638094,6973808,4301305,ADRIANA FERNANDA DE MORAIS,2009-10-04,RUA JOAO INACIO SULZBACH,574,,INDUSTRIAS,4307807.0,RS,95880000,51997359224,,gabriel-6973808@estudante.rs.gov.br,100,GABRIEL MORAIS GONCALVES,05531638094,2009-10-04,23662803107,S/N,0,ALUNO
501,KALYEL DORNELLES BERKAIER,KALYEL BERKAIER,06125171019,6921837,4309209,CAMILA ANDREADORNELLES REHBLEIN,2009-05-20,RUA GOIAS,298,,FATIMA,4303103.0,RS,94955180,51985117948,,kalyel-6921837@estudante.rs.gov.br,0,KALYEL DORNELLES BERKAIER,06125171019,2009-05-20,23707907690,NaN,0,ALUNO


In [79]:
final.isna().sum()

NOME COMPLETO         0
NOME CARTÃO           0
COD_CPF               0
MATRICULA             0
SETOR                 0
NOME_MAE              0
DTA_NASC              0
LOGRADOURO            0
NUMERO                0
COMPLEMENTO           0
BAIRRO                0
CIDADE                0
ESTADO(UF)            0
CEP                   0
TELEFONE1             0
TELEFONE2             0
EMAIL                17
RENDA_MEDIA           0
NOME_RESP             0
CPF_RESP              0
DTA_NASC_RESP         0
COD_NIS               0
RG                14985
MAIOR DE IDADE        0
ORIGEM_CPF            0
dtype: int64

# Arquivos Finais Base

**formata arquivo banri:**

In [80]:
final_banri = final.copy()
final_banri = final_banri[['COD_CPF', 'NOME COMPLETO', 'DTA_NASC', 'NOME_MAE','SETOR',
        'LOGRADOURO', 'NUMERO', 'BAIRRO', 'COMPLEMENTO',
       'CEP', 'TELEFONE1', 'TELEFONE2', 'EMAIL', 'RENDA_MEDIA', 'ORIGEM_CPF', 
       'CPF_RESP', 'NOME_RESP',  'DTA_NASC_RESP', 'RG', 'COD_NIS', 'CIDADE']]
final_banri = final_banri.rename(columns={'NOME COMPLETO': 'NOME_BENEFICIARIO', 'DTA_NASC': 'DT_NASC_BENEFICIARIO', 
                                          'NOME_MAE': 'NOME_MAE_BENEFICIARIO', 'SETOR': 'COD_MUNIC_IBGE', 
                                          'NUMERO': 'NUMERO_LOGRAD', 'BAIRRO': 'NOME_BAIRRO', 'COMPLEMENTO': 'COMPLEMENTO_LOGRAD', 
                                          'TELEFONE1': 'NRO_FONE1', 'TELEFONE2': 'NRO_FONE2', 'RENDA_MEDIA': 'VLR_RENDA_MEDIA', 
                                          'DTA_NASC_RESP': 'DT_NASC_RESP', 'RG': 'COD_RG'})
# divide os pagamentos em 2 grupos, os que possuem cpf duplicados, e os que não possuem
final_banri_cpf_unico = final_banri[~final_banri.duplicated(subset='COD_CPF', keep=False)]
final_banri = final_banri[final_banri.duplicated(subset='COD_CPF', keep=False)]

# atualiza matriculas, email e setor, substituindo pelo valor concatenado de todas ocorrências então dropa duplicatas
final_banri['EMAIL'] = ''
final_banri['COD_MUNIC_IBGE'] = final_banri['CIDADE']
final_banri['COD_MUNIC_IBGE'] = final_banri['COD_MUNIC_IBGE'].astype("int64")
final_banri = final_banri.drop_duplicates()

# concatena duplicados e não-duplicados
final_banri = pd.concat([final_banri, final_banri_cpf_unico], axis=0)
final_banri = final_banri.drop(columns='CIDADE').rename(columns={'CPF_RESP': 'COD_CPF_RESP'})
if salva:
    final_banri.to_csv(Arquivo_Final + f'CPF ÚNICOS {data}_TJE_Base_Beneficiários_Mapeados_{sigla_mes}_2025.csv', index=False, sep=';')
base_banri = final_banri.copy()
final_banri

,COD_CPF,NOME_BENEFICIARIO,DT_NASC_BENEFICIARIO,NOME_MAE_BENEFICIARIO,COD_MUNIC_IBGE,LOGRADOURO,NUMERO_LOGRAD,NOME_BAIRRO,COMPLEMENTO_LOGRAD,CEP,NRO_FONE1,NRO_FONE2,EMAIL,VLR_RENDA_MEDIA,ORIGEM_CPF,COD_CPF_RESP,NOME_RESP,DT_NASC_RESP,COD_RG,COD_NIS
8167,04009923024,TALITA GARCIA MEIRELES,1997-01-22,SIMONE GARCIA MEIRELES,4313375,RUA PITANGUEIRAS,117,CENTRO,,92480000,51998523865,,,0,RESPONSAVEL,04009923024,TALITA GARCIA MEIRELES,1997-01-22,00000000001122916305,20768089691
12567,04054479065,BRENDA CAROLINA MAIDANA DA SILVA,1996-04-07,CARLA MAIDANA,4310207,RUA DARI TISSOTI,241,THOME DE SOUZA,,98700000,54991265029,,,0,RESPONSAVEL,04054479065,BRENDA CAROLINA MAIDANA DA SILVA,1996-04-07,1119117461,20670488415
13059,05137976024,SIMONE DA SILVA MENDES,2001-02-01,SANDRA AMALIA LOPES CARVALHO,4314100,RUA LIBERATO SALZANO,55,SAO CRISTOVAO,FUNDOS CASA 2,99064050,55999101514,,,75,RESPONSAVEL,05137976024,SIMONE DA SILVA MENDES,2001-02-01,8115063946,16629434553
16398,04352845094,GRACIELE LEAL RAMOS,1996-08-01,MARIA CRISTINA LEAL RAMOS,4316402,ESTRADA RINCAO DOS NEGROS,S/N,PICADAS,SN CASA,97590000,55999573097,,,444,RESPONSAVEL,04352845094,GRACIELE LEAL RAMOS,1996-08-01,8126130833,21056093392
25347,04492904093,AMANDA DOS SANTOS AJALA,2002-03-18,GLECI CAMARGO DOS SANTOS,4316402,RUA CACEQUI,1199,LAGOA,CASA,97590000,55998424663,,,50,RESPONSAVEL,04492904093,AMANDA DOS SANTOS AJALA,2002-03-18,00000000007127394661,21215081156
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,05988221009,MARIANE DE AVILA DE LIMA,2009-07-10,PRISCILA NEITZKE DE AVILA,4304663,RUA LUIZ MARIA DA ROCHA,128,JARDIM AMERICA,,96160000,53991735119,,mariane-7050459@estudante.rs.gov.br,0,ALUNO,05988221009,MARIANE DE AVILA DE LIMA,2009-07-10,S/N,23790521066
499,07022746079,ELISIANE SANTOS CUNHA,2007-10-10,CARMEM LIOMAR DOS SANTOS E SANTOS,4303509,RUA EDERALDO DE SOUZA GOMES,12,GETLIO VARGAS,,96785000,51989305983,,elisiane-6957477@estudante.rs.gov.br,338,ALUNO,07022746079,ELISIANE SANTOS CUNHA,2007-10-10,1137719281,21224567716
500,05531638094,GABRIEL MORAIS GONCALVES,2009-10-04,ADRIANA FERNANDA DE MORAIS,4301305,RUA JOAO INACIO SULZBACH,574,INDUSTRIAS,,95880000,51997359224,,gabriel-6973808@estudante.rs.gov.br,100,ALUNO,05531638094,GABRIEL MORAIS GONCALVES,2009-10-04,S/N,23662803107
501,06125171019,KALYEL DORNELLES BERKAIER,2009-05-20,CAMILA ANDREADORNELLES REHBLEIN,4309209,RUA GOIAS,298,FATIMA,,94955180,51985117948,,kalyel-6921837@estudante.rs.gov.br,0,ALUNO,06125171019,KALYEL DORNELLES BERKAIER,2009-05-20,NaN,23707907690


In [81]:
mapeados_final = final.copy()
final = final[['ORIGEM_CPF', 'NOME COMPLETO', 'NOME CARTÃO', 'COD_CPF', 'MATRICULA', 'SETOR',
       'NOME_MAE', 'MAIOR DE IDADE', 'DTA_NASC', 'LOGRADOURO', 'NUMERO', 'COMPLEMENTO', 'BAIRRO',
       'CIDADE', 'ESTADO(UF)', 'CEP', 'TELEFONE1', 'EMAIL', 'RENDA_MEDIA',
       'NOME_RESP', 'CPF_RESP', 'DTA_NASC_RESP']].rename(columns={'TELEFONE1': 'TELEFONE'})
if salva:
    salva_com_auto_fit_colunas(final, Arquivo_Final + f'{data}_TJE_Base_Beneficiários_Mapeados_{sigla_mes}_2025.xlsx', 'responsaveis')
print(str(len(final['COD_CPF'].unique())) + ' CPFs únicos foram mapeados.') # 111692
print(str(len(final['MATRICULA'].unique())) + ' matrículas únicas foram mapeadas.') # 111925
final

101682 CPFs únicos foram mapeados.
101850 matrículas únicas foram mapeadas.


,ORIGEM_CPF,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,MAIOR DE IDADE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP
8167,RESPONSAVEL,TALITA GARCIA MEIRELES,TALITA MEIRELES,04009923024,2992058,4304606,SIMONE GARCIA MEIRELES,1,1997-01-22,RUA PITANGUEIRAS,117,,CENTRO,4313375.0,RS,92480000,51998523865,talita-2992058@estudante.rs.gov.br,0,TALITA GARCIA MEIRELES,04009923024,1997-01-22
12567,RESPONSAVEL,BRENDA CAROLINA MAIDANA DA SILVA,BRENDA SILVA,04054479065,187776,4322509,CARLA MAIDANA,1,1996-04-07,RUA DARI TISSOTI,241,,THOME DE SOUZA,4310207.0,RS,98700000,54991265029,brenda-187776@estudante.rs.gov.br,0,BRENDA CAROLINA MAIDANA DA SILVA,04054479065,1996-04-07
13059,RESPONSAVEL,SIMONE DA SILVA MENDES,SIMONE MENDES,05137976024,5371242,4300406,SANDRA AMALIA LOPES CARVALHO,1,2001-02-01,RUA LIBERATO SALZANO,55,FUNDOS CASA 2,SAO CRISTOVAO,4314100.0,RS,99064050,55999101514,simone-5371242@estudante.rs.gov.br,75,SIMONE DA SILVA MENDES,05137976024,2001-02-01
16398,RESPONSAVEL,GRACIELE LEAL RAMOS,GRACIELE RAMOS,04352845094,2886708,4316402,MARIA CRISTINA LEAL RAMOS,1,1996-08-01,ESTRADA RINCAO DOS NEGROS,S/N,SN CASA,PICADAS,4316402.0,RS,97590000,55999573097,graciele-lramos@estudante.rs.gov.br,444,GRACIELE LEAL RAMOS,04352845094,1996-08-01
25347,RESPONSAVEL,AMANDA DOS SANTOS AJALA,AMANDA AJALA,04492904093,662091,4316402,GLECI CAMARGO DOS SANTOS,1,2002-03-18,RUA CACEQUI,1199,CASA,LAGOA,4316402.0,RS,97590000,55998424663,amanda-662091@estudante.rs.gov.br,50,AMANDA DOS SANTOS AJALA,04492904093,2002-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,ALUNO,MARIANE DE AVILA DE LIMA,MARIANE LIMA,05988221009,7050459,4304663,PRISCILA NEITZKE DE AVILA,0,2009-07-10,RUA LUIZ MARIA DA ROCHA,128,,JARDIM AMERICA,4304663.0,RS,96160000,53991735119,mariane-7050459@estudante.rs.gov.br,0,MARIANE DE AVILA DE LIMA,05988221009,2009-07-10
499,ALUNO,ELISIANE SANTOS CUNHA,ELISIANE CUNHA,07022746079,6957477,4303509,CARMEM LIOMAR DOS SANTOS E SANTOS,0,2007-10-10,RUA EDERALDO DE SOUZA GOMES,12,,GETLIO VARGAS,4303509.0,RS,96785000,51989305983,elisiane-6957477@estudante.rs.gov.br,338,ELISIANE SANTOS CUNHA,07022746079,2007-10-10
500,ALUNO,GABRIEL MORAIS GONCALVES,GABRIEL GONCALVES,05531638094,6973808,4301305,ADRIANA FERNANDA DE MORAIS,0,2009-10-04,RUA JOAO INACIO SULZBACH,574,,INDUSTRIAS,4307807.0,RS,95880000,51997359224,gabriel-6973808@estudante.rs.gov.br,100,GABRIEL MORAIS GONCALVES,05531638094,2009-10-04
501,ALUNO,KALYEL DORNELLES BERKAIER,KALYEL BERKAIER,06125171019,6921837,4309209,CAMILA ANDREADORNELLES REHBLEIN,0,2009-05-20,RUA GOIAS,298,,FATIMA,4303103.0,RS,94955180,51985117948,kalyel-6921837@estudante.rs.gov.br,0,KALYEL DORNELLES BERKAIER,06125171019,2009-05-20


In [82]:
final[final['MATRICULA'] == 7020008]

,ORIGEM_CPF,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,MAIOR DE IDADE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP
78793,ALUNO,MATHEUS RUBATTINO SCOUTO,MATHEUS SCOUTO,86739662053,7020008,4323002,FERNANDA VIANA RUBATTINO,0,2008-10-07,RUA CONCEICAO,474,FRENTE,CECILIA,4323002.0,RS,94475590,51990195977,matheus-7020008@estudante.rs.gov.br,0,FERNANDA VIANA RUBATTINO,82840334020,1981-05-18


In [83]:
query_alunos_tipo_ens = f"""
SELECT UNIQUE IAT.NRO_INT_ALUNO_ESTADO AS MATRICULA, T.CD_TIPO_ENSINO
FROM PDP_SE_STG.ISE_ALUNO_TURMA IAT
INNER JOIN PDP_SE_STG.ISE_TURMA T ON T.NRO_INT_TURMA = IAT.NRO_INT_TURMA
INNER JOIN PDP_SE_STG.ISE_CALENDARIO_ESTAB CE ON CE.NRO_INT_CALEND_ESTAB = IAT.NRO_INT_CALEND_ESTAB AND CURRENT DATE + 5 BETWEEN CE.DT_INICIO_ATIV AND CE.DT_FIM_ATIV
INNER JOIN PDP_SE_STG.ISE_ESTABELECIMENTO IE ON IAT.IDT_ESTAB = IE.IDT_ESTAB AND IE.IN_ESC_PRISIONAL <> 'S' AND IE.NRO_INT_DESIGNACAO NOT IN (10, 489)
INNER JOIN PDP_SE_STG.ISE_ALUNO IA ON IAT.NRO_INT_ALUNO_ESTADO = IA.NRO_INT_ALUNO_ESTADO
WHERE IAT.NRO_INT_SITUACAO IN (1,8)
AND ( 
	(T.CD_TIPO_ENSINO IN ('I2', 'E2', 'R2') -- SE FOR EM
	AND (INTEGER((DAYS(CURRENT DATE) - DAYS(IA.DT_NASCIMENTO)) / 365.25) <= 21)) -- TEM QUE TER 21 ANOS OU MENOS
	OR 
	(T.CD_TIPO_ENSINO IN ('S2', 'E9') -- SE FOR EJA
	AND (INTEGER((DAYS(CURRENT DATE) - DAYS(IA.DT_NASCIMENTO)) / 365.25) <= 29)) -- TEM QUE TER 29 ANOS OU MENOS
	)
"""

alunos_tipo_ens = upper_columns(consulta_datalake(query_alunos_tipo_ens))
final_por_tipo_ens = pd.merge(final[['MATRICULA']], alunos_tipo_ens)
print(final_por_tipo_ens['CD_TIPO_ENSINO'].value_counts())

CD_TIPO_ENSINO
R2    93838
S2     4877
I2     3130
E2       22
Name: count, dtype: int64


In [99]:
query_ds_tipo_ensino = f"""
SELECT CD_TIPO_ENSINO, DS_TIPO_ENSINO
FROM PDP_SE_STG.ISE_TIPO_ENSINO
"""

ds_tipo_ensino = upper_columns(consulta_datalake(query_ds_tipo_ensino))
final_por_tipo_ens = pd.merge(final_por_tipo_ens, ds_tipo_ensino)
print(final_por_tipo_ens['DS_TIPO_ENSINO'].value_counts())

DS_TIPO_ENSINO
Ensino Médio                              93838
EJA Ensino Médio                           4877
Educ Profiss Integrada ao Ensino Médio     3130
Educação Especial Ensino Médio               22
Name: count, dtype: int64


In [84]:
final.isna().sum()

ORIGEM_CPF         0
NOME COMPLETO      0
NOME CARTÃO        0
COD_CPF            0
MATRICULA          0
SETOR              0
NOME_MAE           0
MAIOR DE IDADE     0
DTA_NASC           0
LOGRADOURO         0
NUMERO             0
COMPLEMENTO        0
BAIRRO             0
CIDADE             0
ESTADO(UF)         0
CEP                0
TELEFONE           0
EMAIL             17
RENDA_MEDIA        0
NOME_RESP          0
CPF_RESP           0
DTA_NASC_RESP      0
dtype: int64

**Formato DICMS:** 

[05/04 11:39] Guilherme Henrique Simionato dos Santos
COD_CPF;COD_NIS;COD_FAMILIAR_FAM;VLR_RENDA_TOTAL_FAM;VLR_RENDA_MEDIA_FAM;IND_PBF

In [85]:
from datetime import datetime
import csv

data_atual = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
if formato_dicms:
    # COD_FAM	VLR_RENDA_MEDIA	VLR_RENDA_TOTAL	IND_PAB
    cad_formato_dicms = ler_cad(colunas=['COD_CPF', 'COD_NIS', 'COD_FAM', 'VLR_RENDA_MEDIA', 'VLR_RENDA_TOTAL', 'IND_PAB'])
    cad_formato_dicms = cad_formato_dicms.dropna(subset='COD_CPF')
    base_cpfs = final[['COD_CPF']].drop_duplicates()
    base_cpfs['COD_CPF'] = base_cpfs['COD_CPF'].astype("int64")

    # pega cod_fam do cpf titular
    base_cod_fam = df_cad[df_cad['COD_CPF'].isin(base_cpfs['COD_CPF'].unique())][['COD_FAM']].rename(columns={'COD_CPF': 'COD_FAM'}).drop_duplicates()
    
    # pega rf=1 ou menor rf para cada cod_fam encontrado
    base_cpfs = pd.merge(base_cod_fam, base_cpf_cod_fam).drop(columns='COD_FAM')

    # mescla com as colunas do cad filtrado
    arquivo_formato_dicms = pd.merge(base_cpfs, cad_formato_dicms)
    arquivo_formato_dicms = arquivo_formato_dicms.rename(columns={'COD_FAM': 'COD_FAMILIAR_FAM', 'VLR_RENDA_TOTAL': 'VLR_RENDA_TOTAL_FAM', 
                                                                  'VLR_RENDA_MEDIA': 'VLR_RENDA_MEDIA_FAM', 'IND_PAB': 'IND_PBF'})
    errados = arquivo_formato_dicms[arquivo_formato_dicms['VLR_RENDA_MEDIA_FAM'] > 660]
    print(errados.shape)
    arquivo_formato_dicms = arquivo_formato_dicms[~arquivo_formato_dicms['COD_CPF'].isin(errados['COD_CPF'].unique())]
    
    # formatar CSV no modelo correto
    arquivo_formato_dicms['VLR_RENDA_TOTAL_FAM'] = arquivo_formato_dicms['VLR_RENDA_TOTAL_FAM'].astype("int64")
    arquivo_formato_dicms['COD_CPF'] = arquivo_formato_dicms['COD_CPF'].apply(lambda x: str(x).zfill(11))
    arquivo_formato_dicms['COD_NIS'] = arquivo_formato_dicms['COD_NIS'].apply(lambda x: '' if pd.isna(x) else str(int(x)))
    arquivo_formato_dicms.to_csv(Arquivo_Final + f'DICMS_SEDUC_BENEFICIARIOS_SEDUC_{data_atual}.csv', index=False, sep=';', quoting=csv.QUOTE_NONE, escapechar='\\')
    display(arquivo_formato_dicms)
    
else:
    print('Configurado para não gerar arquivo no formato DICMS')

(0, 6)


,COD_CPF,COD_NIS,COD_FAMILIAR_FAM,VLR_RENDA_MEDIA_FAM,VLR_RENDA_TOTAL_FAM,IND_PBF
0,89617959372,12734889198,11575077,0,0,1
1,67688900000,12394748224,20816448,185,370,1
2,58085904004,12368106709,20853998,506,1518,0
3,70233926020,12446856898,20928408,0,0,1
4,00663864038,16046242429,22146601,303,1515,0
...,...,...,...,...,...,...
95260,01434026094,,20192939270,449,1796,0
95261,01912049007,,20193159473,343,1715,0
95262,00083654070,,20193231506,0,0,0
95263,05683644062,,20193655969,500,500,0


In [86]:
# distribuição de cod_parentesco entre os responsáveis do dicms
aa = arquivo_formato_dicms.copy()
aa['COD_CPF'] = aa['COD_CPF'].astype('int64')
pd.merge(aa, df_cad[['COD_CPF', 'COD_PARENTESCO']])['COD_PARENTESCO'].value_counts()

COD_PARENTESCO
1.0     94763
3.0       202
5.0        27
2.0        17
10.0        4
11.0        2
8.0         2
6.0         1
Name: count, dtype: int64

In [87]:
# quantos dos cpfs que aparecem no arquivo do dicms que são mapeados(maioria absoluta não deve ser)
mapeado = final[['COD_CPF']].drop_duplicates()
mapeado['mapeados'] = True
dicms_comp = pd.merge(arquivo_formato_dicms, mapeado, how='left')
dicms_comp['mapeados'] = dicms_comp['mapeados'].fillna(False)
dicms_comp['mapeados'].value_counts()

/var/folders/fc/pslrc1rs1h7071w42s_lqbrh0000gn/T/ipykernel_5522/3620360818.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dicms_comp['mapeados'] = dicms_comp['mapeados'].fillna(False)


mapeados
False    85506
True      9759
Name: count, dtype: int64

In [88]:
len(arquivo_formato_dicms['COD_CPF'].unique())

95265

In [89]:
arquivo_formato_dicms.isna().sum()

COD_CPF                0
COD_NIS                0
COD_FAMILIAR_FAM       0
VLR_RENDA_MEDIA_FAM    0
VLR_RENDA_TOTAL_FAM    0
IND_PBF                0
dtype: int64

# Arquivos Finais Históricos

In [90]:
df_ultimo_mapeados = pd.read_excel(ultimo_mapeados, sheet_name='responsaveis')#.drop(columns=['Cartão_Cidadão'])

removidos = df_ultimo_mapeados[~df_ultimo_mapeados['MATRICULA'].isin(final['MATRICULA'].unique())]
if salva:
    salva_com_auto_fit_colunas(removidos, Arquivo_Final + f'{data}_TJE_Base_Beneficiários_Inativos_{sigla_mes}_2025.xlsx', 'inativos')
removidos

,ORIGEM_CPF,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,MAIOR DE IDADE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP
48,RESPONSAVEL,CIBELE RAMOS PEREIRA TRINDADE,CIBELE TRINDADE,347940005,2801190,4303103,HELENA RAMOS PEREIRA,1,1982-09-17,RUA JOSE ANGELO BERNARDO,199,,MORADA DO BOSQUE,4303103,RS,94960863,5.198503e+10,isadora-ptrindade@estudante.rs.gov.br,504,CIBELE RAMOS PEREIRA TRINDADE,347940005,1982-09-17
54,RESPONSAVEL,VIVIAN ESKASINKI,VIVIAN ESKASINKI,1883885086,6491469,4314407,CELIA MENEZES ESKASINKI,1,1989-05-01,RUA DOZE BOM JESUS,96,CASA DA FRENTE,AREAL,4314407,RS,96080484,5.398470e+10,maria-6491469@estudante.rs.gov.br,641,VIVIAN ESKASINKI,1883885086,1989-05-01
58,RESPONSAVEL,ROSANI DE OLIVEIRA,ROSANI OLIVEIRA,582841097,4941380,4317509,TEREZINHA LEMES DE OLIVEIRA,1,1975-06-09,RUA RAUL OLIVEIRA,888,AP 1,DITZ,4317509,RS,98802030,5.598117e+10,vitoria-4941380@estudante.rs.gov.br,389,ROSANI DE OLIVEIRA,582841097,1975-06-09
96,RESPONSAVEL,JENIFFER MARTINS KUHN,JENIFFER KUHN,2298770018,6542355,4312401,LEDA MARISA DE ALMEIDA MARTINS,1,1989-07-21,DISTRITO BOA VISTA,S/N,SN,BOA VISTA,4322004,RS,95840000,5.198063e+10,julia-6542355@estudante.rs.gov.br,654,JENIFFER MARTINS KUHN,2298770018,1989-07-21
105,RESPONSAVEL,ANDREIA MACHADO GUERRA,ANDREIA GUERRA,80920268072,3342404,4304606,IRENE TRINDADE MACHADO,1,1982-11-07,RUA SAO LUIS,622,TORRE 5 APT 303,CENTRO,4304606,RS,92310120,5.198540e+10,eduarda-gbonmann@estudante.rs.gov.br,643,ANDREIA MACHADO GUERRA,80920268072,1982-11-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188092,ALUNO,THAINA PEREIRA DE SOUZA,THAINA SOUZA,4733450052,2325567,4321105,MARIA ROSARIA LOPES PEREIRA,1,2006-05-19,RUA ADALTO MORAES,235,,VILA BORGES,4321105,RS,96760000,5.199752e+10,NaN,887,MARIA ROSARIA LOPES PEREIRA DE SOUZA,51615126015,1967-10-08
188093,ALUNO,EDISON LUIS GATEN RAMBO,EDISON RAMBO,3173383079,2815699,4318457,MARCIA ANDREIA GATEN,1,2006-03-30,VILA LINHA ARAUJO,S/N,SN CASA,INTERIOR,4318457,RS,98325000,5.599706e+10,NaN,741,MARCIA ANDREIA GATEN,94769869053,1979-03-30
188094,ALUNO,PERYCLES DAVID CORREA CRUZ,PERYCLES CRUZ,2902243065,2576366,4314902,JULIANA CORREA CRUZ,1,2004-09-18,RUA ARNALDO ANTONIO PORTALUPPI,291,,BOM JESUS,4314902,RS,91420612,5.199897e+10,NaN,1337,JULIANA CORREA CRUZ,1189629062,1985-06-27
188095,ALUNO,WESLEY GABRIEL PACHECO DA SILVA,WESLEY SILVA,5717759002,5608189,4300604,PAMELA CABRAL PACHECO,1,2005-12-19,RUA IRAIDES CASAGRANDE,131,PORTAO,VILA CENTRAL,4300604,RS,94810000,5.199992e+10,NaN,667,PAMELA CABRAL PACHECO,2378489021,1988-12-22


In [91]:
cad_loc = cad_loc[['COD_CPF', 'TELEFONE2', 'RG', 'COD_NIS']]
cad_loc['COD_NIS'] = cad_loc['COD_NIS'].astype(str)
cad_loc['COD_NIS'] = cad_loc['COD_NIS'].str.replace('nan', 'S/N')
cad_loc['COD_NIS'] = cad_loc['COD_NIS'].str.replace('.0', '')
cad_loc['RG'] = cad_loc['RG'].fillna('S/N').str.replace('.0', '')

# removidos formato banri
formato_banri = removidos[['COD_CPF', 'NOME COMPLETO', 'DTA_NASC', 'NOME_MAE','SETOR',
                           'LOGRADOURO', 'NUMERO', 'BAIRRO', 'COMPLEMENTO',
                           'CEP', 'TELEFONE', 'EMAIL', 'RENDA_MEDIA', 'ORIGEM_CPF', 
                           'CPF_RESP', 'NOME_RESP',  'DTA_NASC_RESP', 'CIDADE']]
formato_banri = formato_banri.rename(columns={'NOME COMPLETO': 'NOME_BENEFICIARIO', 'DTA_NASC': 'DT_NASC_BENEFICIARIO', 
                                          'NOME_MAE': 'NOME_MAE_BENEFICIARIO', 'SETOR': 'COD_MUNIC_IBGE', 
                                          'NUMERO': 'NUMERO_LOGRAD', 'BAIRRO': 'NOME_BAIRRO', 'COMPLEMENTO': 'COMPLEMENTO_LOGRAD', 
                                          'TELEFONE': 'NRO_FONE1', 'RENDA_MEDIA': 'VLR_RENDA_MEDIA', 
                                          'DTA_NASC_RESP': 'DT_NASC_RESP', 'CPF_RESP': 'COD_CPF_RESP'})
final_banri = pd.merge(formato_banri, cad_loc).rename(columns={'TELEFONE2': 'NRO_FONE2', 'RG': 'COD_RG'})

# concatena base do banri com os removidos no formato banri já
base_banri['CIDADE'] = base_banri['COD_MUNIC_IBGE']

# garante que colunas das bases estão com o mesmo tipo
base_banri['COD_CPF'] = base_banri['COD_CPF'].astype("int64")
final_banri['COD_CPF'] = final_banri['COD_CPF'].astype("int64")
final_banri['COD_CPF_RESP'] = final_banri['COD_CPF_RESP'].astype("int64").astype(str).str.zfill(11)
final_banri['NRO_FONE1'] = final_banri['NRO_FONE1'].astype(str).str.replace('nan', '').str.replace('.0', '')

final_banri = pd.concat([final_banri, base_banri])
# divide os pagamentos em 2 grupos, os que possuem cpf duplicados, e os que não possuem
final_banri_cpf_unico = final_banri[~final_banri.duplicated(subset='COD_CPF', keep=False)]
final_banri = final_banri[final_banri.duplicated(subset='COD_CPF', keep=False)]

# atualiza matriculas, email e setor, substituindo pelo valor concatenado de todas ocorrências então dropa duplicatas
final_banri['EMAIL'] = ''
final_banri['COD_MUNIC_IBGE'] = final_banri['CIDADE']
final_banri['COD_MUNIC_IBGE'] = final_banri['COD_MUNIC_IBGE'].astype("int64")
final_banri = final_banri.drop_duplicates(subset='COD_CPF', keep='first')

# concatena duplicados e não-duplicados
final_banri = pd.concat([final_banri, final_banri_cpf_unico], axis=0)
formato_banri = final_banri.drop(columns='CIDADE')
formato_banri = formato_banri[['COD_CPF', 'NOME_BENEFICIARIO', 'DT_NASC_BENEFICIARIO',
       'NOME_MAE_BENEFICIARIO', 'COD_MUNIC_IBGE', 'LOGRADOURO',
       'NUMERO_LOGRAD', 'NOME_BAIRRO', 'COMPLEMENTO_LOGRAD', 'CEP',
       'NRO_FONE1', 'NRO_FONE2', 'EMAIL', 'VLR_RENDA_MEDIA', 'ORIGEM_CPF', 'COD_CPF_RESP',
       'NOME_RESP', 'DT_NASC_RESP', 'COD_RG', 'COD_NIS']]

if salva:
    formato_banri.to_csv(Arquivo_Final + f'CPF ÚNICOS {data}_TJE_Base_Acumulada_Cartões_Ativos_{sigla_mes}_2025.csv', index=False, sep=';')
print(str(formato_banri['COD_CPF'].nunique()) + ' CPFs únicos')
formato_banri

174193 CPFs únicos


,COD_CPF,NOME_BENEFICIARIO,DT_NASC_BENEFICIARIO,NOME_MAE_BENEFICIARIO,COD_MUNIC_IBGE,LOGRADOURO,NUMERO_LOGRAD,NOME_BAIRRO,COMPLEMENTO_LOGRAD,CEP,NRO_FONE1,NRO_FONE2,EMAIL,VLR_RENDA_MEDIA,ORIGEM_CPF,COD_CPF_RESP,NOME_RESP,DT_NASC_RESP,COD_RG,COD_NIS
4,78514282034,CRISTIANA DA SILVA,1978-02-21,MARIA BALBINA DA SILVA,4304606,RUA MARIA FAUSTINO CORREA,55,GUAJUVIRAS,FUNDOS SAO JOAO QUADRA A,92440710,51991031856,,,363,RESPONSAVEL,78514282034,CRISTIANA DA SILVA,1978-02-21,NaN,12510356757
6,71949496015,RICARDO ALVES ARISPE,1972-03-24,DELCIA ALVES,4301602,RUA SETECENTOS E 40,450,PRADO VELHO,AP 0001,96418650,53999003575,,,637,RESPONSAVEL,71949496015,RICARDO ALVES ARISPE,1972-03-24,1091518579,12453282818
11,3067900090,JANAINA DA SILVA ALVES,1991-10-01,DINA MARGARETE DA SILVA ALVES,4318309,RUA LUCILIA FAGUNDES,56,BELA VISTA,CASA FUNDOS,97304466,55999532828,,,379,RESPONSAVEL,03067900090,JANAINA DA SILVA ALVES,1991-10-01,00000000005101910122,16199816286
21,40578976072,TALMA LIMA VALLES,1960-12-26,SUELI MARQUES,4316402,RUA GENERAL OSORIO,1844,CENTRO,CASA,97590000,5532313964,,,414,RESPONSAVEL,40578976072,TALMA LIMA VALLES,1960-12-26,S/N,16553198676
41,60153160047,CRISTINA SANTANA DA SILVA,1999-12-15,ELISA APARECIDA DE GOES,4317103,ESTRADA ASSENTAMENTO ANGELO,1900,ITAQUATIA,,97584899,55997059560,,,33,RESPONSAVEL,60153160047,CRISTINA SANTANA DA SILVA,1999-12-15,NaN,16414437620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,5988221009,MARIANE DE AVILA DE LIMA,2009-07-10,PRISCILA NEITZKE DE AVILA,4304663,RUA LUIZ MARIA DA ROCHA,128,JARDIM AMERICA,,96160000,53991735119,,mariane-7050459@estudante.rs.gov.br,0,ALUNO,05988221009,MARIANE DE AVILA DE LIMA,2009-07-10,S/N,23790521066
499,7022746079,ELISIANE SANTOS CUNHA,2007-10-10,CARMEM LIOMAR DOS SANTOS E SANTOS,4303509,RUA EDERALDO DE SOUZA GOMES,12,GETLIO VARGAS,,96785000,51989305983,,elisiane-6957477@estudante.rs.gov.br,338,ALUNO,07022746079,ELISIANE SANTOS CUNHA,2007-10-10,1137719281,21224567716
500,5531638094,GABRIEL MORAIS GONCALVES,2009-10-04,ADRIANA FERNANDA DE MORAIS,4301305,RUA JOAO INACIO SULZBACH,574,INDUSTRIAS,,95880000,51997359224,,gabriel-6973808@estudante.rs.gov.br,100,ALUNO,05531638094,GABRIEL MORAIS GONCALVES,2009-10-04,S/N,23662803107
501,6125171019,KALYEL DORNELLES BERKAIER,2009-05-20,CAMILA ANDREADORNELLES REHBLEIN,4309209,RUA GOIAS,298,FATIMA,,94955180,51985117948,,kalyel-6921837@estudante.rs.gov.br,0,ALUNO,06125171019,KALYEL DORNELLES BERKAIER,2009-05-20,NaN,23707907690


In [92]:
formato_banri.columns

Index(['COD_CPF', 'NOME_BENEFICIARIO', 'DT_NASC_BENEFICIARIO',
       'NOME_MAE_BENEFICIARIO', 'COD_MUNIC_IBGE', 'LOGRADOURO',
       'NUMERO_LOGRAD', 'NOME_BAIRRO', 'COMPLEMENTO_LOGRAD', 'CEP',
       'NRO_FONE1', 'NRO_FONE2', 'EMAIL', 'VLR_RENDA_MEDIA', 'ORIGEM_CPF',
       'COD_CPF_RESP', 'NOME_RESP', 'DT_NASC_RESP', 'COD_RG', 'COD_NIS'],
      dtype='object')

In [93]:
formato_banri.isna().sum()

COD_CPF                      0
NOME_BENEFICIARIO            0
DT_NASC_BENEFICIARIO         0
NOME_MAE_BENEFICIARIO        0
COD_MUNIC_IBGE               0
LOGRADOURO                   0
NUMERO_LOGRAD                0
NOME_BAIRRO                  0
COMPLEMENTO_LOGRAD           0
CEP                          0
NRO_FONE1                    0
NRO_FONE2                    0
EMAIL                      901
VLR_RENDA_MEDIA              0
ORIGEM_CPF                   0
COD_CPF_RESP                 0
NOME_RESP                    0
DT_NASC_RESP                 0
COD_RG                   26807
COD_NIS                      0
dtype: int64

In [94]:
base_acumulada = pd.concat([final, removidos])
if salva:
    salva_com_auto_fit_colunas(base_acumulada, Arquivo_Final + f'{data}_TJE_Base_Acumulada_Estudantes_Beneficiários_{sigla_mes}_2025.xlsx', sheet_name='responsaveis')
print(str(base_acumulada['MATRICULA'].nunique()) + ' matrículas únicas') # 176645
base_acumulada

189542 matrículas únicas


,ORIGEM_CPF,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,MAIOR DE IDADE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP
8167,RESPONSAVEL,TALITA GARCIA MEIRELES,TALITA MEIRELES,04009923024,2992058,4304606,SIMONE GARCIA MEIRELES,1,1997-01-22,RUA PITANGUEIRAS,117,,CENTRO,4313375.0,RS,92480000,51998523865,talita-2992058@estudante.rs.gov.br,0,TALITA GARCIA MEIRELES,04009923024,1997-01-22
12567,RESPONSAVEL,BRENDA CAROLINA MAIDANA DA SILVA,BRENDA SILVA,04054479065,187776,4322509,CARLA MAIDANA,1,1996-04-07,RUA DARI TISSOTI,241,,THOME DE SOUZA,4310207.0,RS,98700000,54991265029,brenda-187776@estudante.rs.gov.br,0,BRENDA CAROLINA MAIDANA DA SILVA,04054479065,1996-04-07
13059,RESPONSAVEL,SIMONE DA SILVA MENDES,SIMONE MENDES,05137976024,5371242,4300406,SANDRA AMALIA LOPES CARVALHO,1,2001-02-01,RUA LIBERATO SALZANO,55,FUNDOS CASA 2,SAO CRISTOVAO,4314100.0,RS,99064050,55999101514,simone-5371242@estudante.rs.gov.br,75,SIMONE DA SILVA MENDES,05137976024,2001-02-01
16398,RESPONSAVEL,GRACIELE LEAL RAMOS,GRACIELE RAMOS,04352845094,2886708,4316402,MARIA CRISTINA LEAL RAMOS,1,1996-08-01,ESTRADA RINCAO DOS NEGROS,S/N,SN CASA,PICADAS,4316402.0,RS,97590000,55999573097,graciele-lramos@estudante.rs.gov.br,444,GRACIELE LEAL RAMOS,04352845094,1996-08-01
25347,RESPONSAVEL,AMANDA DOS SANTOS AJALA,AMANDA AJALA,04492904093,662091,4316402,GLECI CAMARGO DOS SANTOS,1,2002-03-18,RUA CACEQUI,1199,CASA,LAGOA,4316402.0,RS,97590000,55998424663,amanda-662091@estudante.rs.gov.br,50,AMANDA DOS SANTOS AJALA,04492904093,2002-03-18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188092,ALUNO,THAINA PEREIRA DE SOUZA,THAINA SOUZA,4733450052,2325567,4321105,MARIA ROSARIA LOPES PEREIRA,1,2006-05-19,RUA ADALTO MORAES,235,,VILA BORGES,4321105.0,RS,96760000,51997519284.0,NaN,887,MARIA ROSARIA LOPES PEREIRA DE SOUZA,51615126015,1967-10-08
188093,ALUNO,EDISON LUIS GATEN RAMBO,EDISON RAMBO,3173383079,2815699,4318457,MARCIA ANDREIA GATEN,1,2006-03-30,VILA LINHA ARAUJO,S/N,SN CASA,INTERIOR,4318457.0,RS,98325000,55997059731.0,NaN,741,MARCIA ANDREIA GATEN,94769869053,1979-03-30
188094,ALUNO,PERYCLES DAVID CORREA CRUZ,PERYCLES CRUZ,2902243065,2576366,4314902,JULIANA CORREA CRUZ,1,2004-09-18,RUA ARNALDO ANTONIO PORTALUPPI,291,,BOM JESUS,4314902.0,RS,91420612,51998973265.0,NaN,1337,JULIANA CORREA CRUZ,1189629062,1985-06-27
188095,ALUNO,WESLEY GABRIEL PACHECO DA SILVA,WESLEY SILVA,5717759002,5608189,4300604,PAMELA CABRAL PACHECO,1,2005-12-19,RUA IRAIDES CASAGRANDE,131,PORTAO,VILA CENTRAL,4300604.0,RS,94810000,51999915573.0,NaN,667,PAMELA CABRAL PACHECO,2378489021,1988-12-22


In [95]:
a = base_acumulada[['COD_CPF']]
a['COD_CPF'] = a['COD_CPF'].astype("int64")
continuam = pd.merge(a, df_ultimo_mapeados[['COD_CPF']]).drop_duplicates()
cpfs_novos = a[~a['COD_CPF'].isin(continuam['COD_CPF'].unique())]
print(str(cpfs_novos['COD_CPF'].nunique()) + ' cartões novos(podem ser menos, pois alguns novos podem já possuir DICMS)')
if cpfs_novos['COD_CPF'].nunique() < 100:
    print(cpfs_novos)

2396 cartões novos(podem ser menos, pois alguns novos podem já possuir DICMS)


In [96]:
# se deus for bom resulta da igual(número de matrículas que mudaram de cpf)
matriculas_que_deveria_aparecer = df_ultimo_mapeados[~df_ultimo_mapeados['COD_CPF'].isin(continuam['COD_CPF'].unique())]
print(matriculas_que_deveria_aparecer['MATRICULA'].nunique())
b = df_ultimo_mapeados[df_ultimo_mapeados['MATRICULA'].isin(matriculas_que_deveria_aparecer['MATRICULA'].unique())]
print(b['MATRICULA'].nunique())
b

977
977


,ORIGEM_CPF,NOME COMPLETO,NOME CARTÃO,COD_CPF,MATRICULA,SETOR,NOME_MAE,MAIOR DE IDADE,DTA_NASC,LOGRADOURO,NUMERO,COMPLEMENTO,BAIRRO,CIDADE,ESTADO(UF),CEP,TELEFONE,EMAIL,RENDA_MEDIA,NOME_RESP,CPF_RESP,DTA_NASC_RESP
16,RESPONSAVEL,ELISANGELA ABDALA VARGAS,ELISANGELA VARGAS,544152093,5300422,4300604,ELZA VARGAS RIBEIRO,1,1983-07-24,RUA JOSE LINS DO REGO,850,,MARIA REGINA,4300604,RS,94828460,5.198916e+10,joao-vvaires@estudante.rs.gov.br,253,ELISANGELA ABDALA VARGAS,544152093,1983-07-24
18,RESPONSAVEL,VANESSA SANTOS CARDOZO,VANESSA CARDOZO,81107277000,4597384,4305454,MARIA DOLORES SANTOS CARDOZO,1,1982-04-17,RUA SAMAMBAIA,222,,LOMBA DO PINHEIRO,4314902,RS,91550232,5.198183e+10,matheus-erodrigues@estudante.rs.gov.br,0,VANESSA SANTOS CARDOZO,81107277000,1982-04-17
27,RESPONSAVEL,ZILDA RUSCH,ZILDA RUSCH,34278095015,5324579,4314407,HOLINDA LINDMAN RUSCH,1,1959-03-09,RUA DOM PEDRO II,257,,CENTRO,4314407,RS,96010300,5.398103e+10,ana-cldsantos5@estudante.rs.gov.br,506,ZILDA RUSCH,34278095015,1959-03-09
31,RESPONSAVEL,RAQUEL LEMES,RAQUEL LEMES,1514140098,2508705,4304358,HILDA DOS SANTOS CHAGAS,1,1987-10-05,ESTRADA NOSSA SENHORA APARECIDA,S/N,SN,ASSENTAMENTO,4304358,RS,96495000,5.399911e+10,ariele-2508705@estudante.rs.gov.br,379,RAQUEL LEMES,1514140098,1987-10-05
34,RESPONSAVEL,ALESSANDRA DA SILVA PEREIRA,ALESSANDRA PEREIRA,1371653070,3766166,4314407,NEIVA GOUVEA DA SILVA,1,1985-12-08,RUA SEIS,337,CASA FUNDOS,AREAL,4314407,RS,96081175,5.398134e+10,tayla-3766166@estudante.rs.gov.br,0,ALESSANDRA DA SILVA PEREIRA,1371653070,1985-12-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181871,RESPONSAVEL,ANGELICA FREITAS LUCAS DA SILVA,ANGELICA SILVA,53760735053,6544818,4304663,MARIA ZILA FREITAS LUCAS,1,1970-09-30,RUA LUIS MARIA DA ROCHA,1353,FUNDOS,JARDIM AMERICA,4304663,RS,96160000,5.398462e+10,gabriele-6544818@educar.rs.gov.br,790,ANGELICA FREITAS LUCAS DA SILVA,53760735053,1970-09-30
181893,RESPONSAVEL,ROSELAINE MATOS DOS SANTOS,ROSELAINE SANTOS,76944433087,2610307,4309209,PRUDENCIA MATOS DOS SANTOS,1,1974-11-29,RUA JARDEL FILHO,819,,AGUAS CLARAS,4309209,RS,94110311,5.134976e+09,kayan-2610307@educar.rs.gov.br,1152,ROSELAINE MATOS DOS SANTOS,76944433087,1974-11-29
183790,ALUNO,ANA CAROLINA PERLAKY,ANA PERLAKY,87011522034,6703086,4311403,TATIANE PERLAKY,0,2008-10-26,RUA MARECHAL DEODORO,674,,CENTRO,4311403,RS,95900124,5.199875e+10,ana-cperlaky@educar.rs.gov.br,0,ELISABETE ZENI KOPP,65487060053,1970-01-27
185606,ALUNO,ANA CLARA GARCIA OLIVEIRA,ANA OLIVEIRA,12388062950,1651655,4318705,DAIANA ELCY RODRIGUES GARCIA,1,2006-05-18,RUA DAS ABELHAS,120,,HORTO FLORESTAL,4320008,RS,93212605,5.198949e+10,ana-cgoliveira@educar.rs.gov.br,63,DAIANA ELCY RODRIGUES GARCIA,1103871048,1985-08-30


In [97]:
df_ultimo_mapeados.isna().sum()

ORIGEM_CPF           0
NOME COMPLETO        0
NOME CARTÃO          0
COD_CPF              0
MATRICULA            0
SETOR                0
NOME_MAE             0
MAIOR DE IDADE       0
DTA_NASC             0
LOGRADOURO           0
NUMERO               0
COMPLEMENTO          0
BAIRRO               0
CIDADE               0
ESTADO(UF)           0
CEP                  0
TELEFONE          6220
EMAIL             2623
RENDA_MEDIA          0
NOME_RESP            0
CPF_RESP             0
DTA_NASC_RESP        0
dtype: int64